In [1]:
# Cell 1: Setup & Configuration
import os
import re
import json
import pickle
import warnings
from pathlib import Path
from dataclasses import dataclass

import pandas as pd
import numpy as np

warnings.filterwarnings("ignore")

import os
os.environ["TQDM_NOTEBOOK_DISABLE"] = "1"

# --- PATHS: adjust this ONE line to your actual folder ---
# Point this to the folder that CONTAINS your ECT/ subfolder and this notebook
PROJECT_ROOT = Path(".").resolve()

# If your .txt files are in a subfolder called "ECT", this will find them:
DATA_DIR = PROJECT_ROOT / "ECT"
CACHE_DIR = PROJECT_ROOT / "cache"
MODELS_DIR = PROJECT_ROOT / "models"

CACHE_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# --- Verify everything looks right ---
txt_files = sorted(DATA_DIR.glob("*.txt")) if DATA_DIR.is_dir() else []

print(f"Project root : {PROJECT_ROOT}")
print(f"Data dir     : {DATA_DIR}")
print(f"Files found  : {len(txt_files)}")

if len(txt_files) == 0:
    print("\n⚠️  NO .txt FILES FOUND!")
    print(f"Looked in: {DATA_DIR}")
    print("Fix: edit the PROJECT_ROOT or DATA_DIR line above.")
    print("Example:")
    print(r'  DATA_DIR = Path(r"C:\Users\loren\Downloads\my_project\ECT")')
else:
    print(f"First 5: {[f.name for f in txt_files[:5]]}")
    print("\n✅ Ready to proceed!")

Project root : C:\Users\loren\Desktop\MTH9796\HW2
Data dir     : C:\Users\loren\Desktop\MTH9796\HW2\ECT
Files found  : 131
First 5: ['AMD_Q1-2024.txt', 'AMD_Q1-2025.txt', 'AMD_Q2-2024.txt', 'AMD_Q2-2025.txt', 'AMD_Q3-2024.txt']

✅ Ready to proceed!


In [2]:
# Cell 2: Install sentence segmenter
try:
    import pysbd
    print("pysbd already installed")
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pysbd", "-q"])
    import pysbd
    print("pysbd installed")

pysbd already installed


In [3]:
# Cell 3: Transcript Parser

SECTION_HEADERS = {
    "Presentation Operator Message": ("presentation", "operator"),
    "Presenter Speech": ("presentation", "executive"),
    "Question and Answer Operator Message": ("qa", "operator"),
    "Question": ("qa", "analyst"),
    "Answer": ("qa", "executive"),
}

SPEAKER_LINE_PATTERN = re.compile(r"^(Operator|Executives|Analysts)(\s*-\s*.+)?$")

_BODY_OPENERS = (
    "Thanks", "Thank", "Yes", "Sure", "Hi", "Hello", "Good", "Hey", "OK", "Okay",
    "Right", "Well", "Maybe", "Actually", "So", "And", "But", "I'll", "We'll",
    "I'd", "We'd", "I'm", "We're", "Let me", "Let's",
)

_INLINE_BODY_PATTERN = re.compile(
    r"^(?P<label>(?:Operator|Executives|Analysts)(?:\s*-\s*[^.?!]+?)?)\s+"
    r"(?P<body>(?:" + "|".join(re.escape(w) for w in _BODY_OPENERS) + r")\b.*)$"
)


@dataclass
class Block:
    section: str
    speaker_role: str
    speaker_label: str
    text: str


def split_label_and_body(line):
    m = _INLINE_BODY_PATTERN.match(line)
    if not m:
        return line, ""
    return m.group("label").strip(), m.group("body").strip()


def _looks_like_speaker_label(line):
    stripped = line.strip()
    if not stripped:
        return False
    if not stripped.startswith(("Operator", "Executives", "Analysts")):
        return False
    if len(stripped) > 200:
        return False
    if stripped.endswith((".", "?", "!")):
        return False
    cleaned = re.sub(
        r"\b(?:U\.S\.|U\.K\.|U\.S\.A\.|Ph\.D\.|M\.D\.|Jr\.|Sr\.|Inc\.|Corp\.|Co\.|Ltd\.|St\.)\s*",
        " ", stripped
    )
    if any(p in cleaned for p in [". ", "? ", "! "]):
        return False
    if SPEAKER_LINE_PATTERN.match(stripped):
        return True
    return False


def parse_filename(filename):
    stem = Path(filename).stem
    m = re.match(r"^([A-Z]+)_Q(\d)-(\d{4})$", stem)
    if not m:
        return {"ticker": stem.split("_")[0], "quarter": None, "fiscal_year": None}
    return {
        "ticker": m.group(1),
        "quarter": f"Q{m.group(2)}",
        "fiscal_year": int(m.group(3)),
    }


def parse_transcript(text):
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    lines = text.split("\n")
    blocks = []
    i, n = 0, len(lines)

    while i < n and lines[i].strip() not in SECTION_HEADERS:
        i += 1

    current_section = None
    current_default_role = None

    while i < n:
        header_or_label = lines[i].strip()

        if header_or_label in SECTION_HEADERS:
            current_section, current_default_role = SECTION_HEADERS[header_or_label]
            i += 1
            while i < n and not lines[i].strip():
                i += 1
            if i >= n:
                break
            speaker_label_line = lines[i].strip()
        elif current_section is not None and _looks_like_speaker_label(header_or_label):
            speaker_label_line = header_or_label
        else:
            i += 1
            continue

        speaker_label = speaker_label_line
        inline_body = ""
        peeled_label, peeled_body = split_label_and_body(speaker_label)
        if peeled_body:
            speaker_label = peeled_label
            inline_body = peeled_body
            i += 1
        elif SPEAKER_LINE_PATTERN.match(speaker_label):
            i += 1
        else:
            speaker_label = current_default_role.title()

        if speaker_label.startswith("Operator"):
            speaker_role = "operator"
        elif speaker_label.startswith("Executives"):
            speaker_role = "executive"
        elif speaker_label.startswith("Analysts"):
            speaker_role = "analyst"
        else:
            speaker_role = current_default_role

        body_lines = []
        if inline_body:
            body_lines.append(inline_body)
        while i < n:
            stripped = lines[i].strip()
            if stripped in SECTION_HEADERS or _looks_like_speaker_label(stripped):
                break
            body_lines.append(lines[i])
            i += 1

        body = "\n".join(body_lines).strip()
        if body:
            blocks.append(Block(
                section=current_section,
                speaker_role=speaker_role,
                speaker_label=speaker_label,
                text=body,
            ))

    return blocks


print("Parser loaded OK")

Parser loaded OK


In [4]:
# Cell 4: Sentence Segmenter

_SEGMENTER = pysbd.Segmenter(language="en", clean=False)
_RESPLIT_PATTERN = re.compile(r"(?<=[.!?])\s+(?=[A-Z])")

_ABBREV = {
    "Mr", "Mrs", "Ms", "Dr", "Sr", "Jr", "Inc", "Corp", "Co", "Ltd",
    "St", "vs", "etc", "e.g", "i.e", "U.S", "U.K", "No",
}


def _is_abbrev_boundary(prev_text):
    m = re.search(r"(\S+)\.$", prev_text.strip())
    if not m:
        return False
    return m.group(1).strip() in _ABBREV


def _resplit_long_sentence(sent, max_len=600):
    if len(sent) <= max_len:
        return [sent]
    parts = _RESPLIT_PATTERN.split(sent)
    if len(parts) == 1:
        return [sent]
    fixed = []
    buf = parts[0]
    for p in parts[1:]:
        if _is_abbrev_boundary(buf):
            buf = buf + " " + p
        else:
            fixed.append(buf)
            buf = p
    fixed.append(buf)
    return [s.strip() for s in fixed if s.strip()]


def segment_sentences(text):
    text = re.sub(r"\s+", " ", text).strip()
    if not text:
        return []
    sents = _SEGMENTER.segment(text)
    out = []
    for s in sents:
        s = s.strip()
        if not s:
            continue
        out.extend(_resplit_long_sentence(s))
    return [s.strip() for s in out if s.strip()]


print("Segmenter loaded OK")

Segmenter loaded OK


In [5]:
# Cell 5: Extract sentences from all transcripts

import tqdm
tqdm_bar = tqdm.tqdm  # force the plain version, skip notebook widget

MIN_CHAR_LENGTH = 40
SENTENCES_CACHE = CACHE_DIR / "sentences.parquet"

if SENTENCES_CACHE.exists():
    print(f"Loading cached sentences from {SENTENCES_CACHE}")
    df_sentences = pd.read_parquet(SENTENCES_CACHE)
else:
    files = sorted(DATA_DIR.glob("*.txt"))
    print(f"Found {len(files)} transcripts — extracting sentences...")

    rows = []
    for fp in tqdm_bar(files, file=__import__('sys').stdout):
        meta = parse_filename(fp.name)
        raw = fp.read_text(encoding="utf-8-sig")
        blocks = parse_transcript(raw)
        for block_idx, block in enumerate(blocks):
            sents = segment_sentences(block.text)
            for sent_idx, sent in enumerate(sents):
                rows.append({
                    "source_file": fp.name,
                    "ticker": meta["ticker"],
                    "quarter": meta["quarter"],
                    "fiscal_year": meta["fiscal_year"],
                    "section": block.section,
                    "speaker_role": block.speaker_role,
                    "speaker_label": block.speaker_label,
                    "block_idx": block_idx,
                    "sentence_idx_in_block": sent_idx,
                    "text": sent,
                    "char_len": len(sent),
                })

    print(f"\nExtracted {len(rows):,} raw sentences")

    if len(rows) == 0:
        raise RuntimeError(
            f"Parser produced 0 sentences from {len(files)} files. "
            f"Check that your .txt files contain headers like 'Presentation Operator Message'."
        )

    df_sentences = pd.DataFrame(rows)

    df_sentences = df_sentences[df_sentences["char_len"] >= MIN_CHAR_LENGTH].copy()
    print(f"After length filter (>= {MIN_CHAR_LENGTH} chars): {len(df_sentences):,}")

    n_before = len(df_sentences)
    df_sentences = df_sentences.drop_duplicates(subset=["text"]).reset_index(drop=True)
    print(f"After dedup: {len(df_sentences):,} (removed {n_before - len(df_sentences):,} duplicates)")

    df_sentences.insert(0, "sentence_id", range(len(df_sentences)))

    df_sentences.to_parquet(SENTENCES_CACHE, index=False)
    print(f"Saved → {SENTENCES_CACHE}")

print(f"\n{'='*50}")
print(f"Total sentences in pool: {len(df_sentences):,}")
print(f"\nBy section:\n{df_sentences['section'].value_counts().to_string()}")
print(f"\nBy speaker role:\n{df_sentences['speaker_role'].value_counts().to_string()}")
print(f"\nSentence length stats:\n{df_sentences['char_len'].describe().round(1).to_string()}")

Loading cached sentences from C:\Users\loren\Desktop\MTH9796\HW2\cache\sentences.parquet

Total sentences in pool: 52,436

By section:
section
qa              31639
presentation    20797

By speaker role:
speaker_role
executive    43780
analyst       7480
operator      1176

Sentence length stats:
count    52436.0
mean       124.2
std         69.2
min         40.0
25%         74.0
50%        109.0
75%        157.0
max       1004.0


In [6]:
# Cell 6: Spot-check samples from each category

print("=== Operator sentences (expect boilerplate) ===")
for _, r in df_sentences[df_sentences.speaker_role == "operator"].sample(5, random_state=42).iterrows():
    print(f"  • {r.text[:140]}")

print("\n=== Analyst Q&A (expect substantive) ===")
for _, r in df_sentences[df_sentences.speaker_role == "analyst"].sample(5, random_state=42).iterrows():
    print(f"  • {r.text[:140]}")

print("\n=== Executive presentation (mixed) ===")
for _, r in df_sentences[(df_sentences.speaker_role == "executive") & (df_sentences.section == "presentation")].sample(5, random_state=42).iterrows():
    print(f"  • {r.text[:140]}")

print("\n=== Executive Q&A (mixed) ===")
for _, r in df_sentences[(df_sentences.speaker_role == "executive") & (df_sentences.section == "qa")].sample(5, random_state=42).iterrows():
    print(f"  • {r.text[:140]}")

=== Operator sentences (expect boilerplate) ===
  • Ladies and gentlemen, due to the interest of time, I would now like to turn the call back over to Ji Yoo for closing remarks.
  • We'll go next to Steven Chubak with Wolfe Research.
  • The next question comes from the line of Stacy Rasgon with Bernstein Research.
  • The next question is from Jim Mitchell with Seaport Global.
  • [Operator Instructions] For our first question, it's coming from the line of Matt O'Connor from Deutsche Bank.

=== Analyst Q&A (expect substantive) ===
  • I know you touched briefly on it, but just maybe unpacking the degree of confidence in the operating leverage over the medium term and then,
  • I understand what you just said about how strong demand is.
  • And congrats again on a really impressive quarter and start to the year.
  • Do your comments on Blackwell imply that we reaccelerate from there as you get more supply?
  • And I'm just curious how you would define success in that external business.

In [7]:
# Cell 7: Sample sentences for gold labeling

GOLD_POOL_SIZE = 2500
GOLD_POOL_CACHE = CACHE_DIR / "gold_pool.parquet"

if GOLD_POOL_CACHE.exists():
    print(f"Loading cached gold pool from {GOLD_POOL_CACHE}")
    df_gold_pool = pd.read_parquet(GOLD_POOL_CACHE)
else:
    # Stratified sample across section x speaker_role
    # This ensures we get enough operator (boilerplate-heavy), analyst (substantive-heavy),
    # and executive (mixed) sentences for a balanced gold set.
    
    df_sentences["stratum"] = df_sentences["section"] + "_" + df_sentences["speaker_role"]
    
    print("Population by stratum:")
    print(df_sentences["stratum"].value_counts().to_string())
    print()
    
    # Sample proportionally, but guarantee minimums for small strata
    MIN_PER_STRATUM = 50
    strata_counts = df_sentences["stratum"].value_counts()
    
    samples = []
    remaining_budget = GOLD_POOL_SIZE
    
    # First pass: guarantee minimum for small strata
    for stratum, pop_count in strata_counts.items():
        n_sample = min(MIN_PER_STRATUM, pop_count)
        stratum_df = df_sentences[df_sentences["stratum"] == stratum]
        samples.append(stratum_df.sample(n=n_sample, random_state=RANDOM_SEED))
        remaining_budget -= n_sample
    
    # Second pass: fill remaining budget proportionally from larger strata
    large_strata = strata_counts[strata_counts > MIN_PER_STRATUM]
    total_large = large_strata.sum()
    
    for stratum, pop_count in large_strata.items():
        n_extra = int(remaining_budget * pop_count / total_large)
        already_sampled = MIN_PER_STRATUM
        stratum_df = df_sentences[df_sentences["stratum"] == stratum]
        already_ids = samples[-len(strata_counts) + list(strata_counts.index).index(stratum)].index
        available = stratum_df.drop(already_ids, errors="ignore")
        n_extra = min(n_extra, len(available))
        if n_extra > 0:
            samples.append(available.sample(n=n_extra, random_state=RANDOM_SEED))
    
    df_gold_pool = pd.concat(samples).drop_duplicates(subset=["sentence_id"]).reset_index(drop=True)
    
    # Drop the helper column
    if "stratum" in df_gold_pool.columns:
        df_gold_pool = df_gold_pool.drop(columns=["stratum"])
    if "stratum" in df_sentences.columns:
        df_sentences = df_sentences.drop(columns=["stratum"])
    
    df_gold_pool.to_parquet(GOLD_POOL_CACHE, index=False)
    print(f"Saved gold pool → {GOLD_POOL_CACHE}")

print(f"\nGold pool size: {len(df_gold_pool):,}")
print(f"\nBy section x speaker_role:")
print(pd.crosstab(df_gold_pool["section"], df_gold_pool["speaker_role"], margins=True))

Loading cached gold pool from C:\Users\loren\Desktop\MTH9796\HW2\cache\gold_pool.parquet

Gold pool size: 2,348

By section x speaker_role:
speaker_role  analyst  executive  operator   All
section                                         
presentation        0        881        50   931
qa                320       1047        50  1417
All               320       1928       100  2348


In [8]:

# Cell 8: Labeling rubric and judge prompts (Anthropic + OpenAI + Mistral)

LABELING_RUBRIC = """
You are labeling sentences from earnings-call transcripts as either BOILERPLATE or SUBSTANTIVE.

BOILERPLATE — scripted, generic, or procedural language that carries no material information:
  • Safe-harbor / forward-looking statement disclaimers
  • Operator instructions ("Please press star-one to ask a question")
  • Introducing the next speaker by name/title ("Our next question comes from X at Y")
  • Generic greetings and thanks ("Thank you for joining", "Good morning everyone")
  • Passing the call ("I'll turn it over to Jean", "Let me hand it back to the operator")
  • One-word or minimal acknowledgments ("Yes", "Sure", "Absolutely", "Thanks")
  • Boilerplate meeting logistics ("This call is being recorded", "A replay will be available")

SUBSTANTIVE — contains material information an analyst or investor would want to read:
  • Revenue, earnings, margins, or any specific financial figures
  • Forward guidance, outlook, or forecasts (even qualitative ones like "we expect growth")
  • Segment-level commentary ("Data Center revenue grew 38%")
  • Strategy, competitive positioning, product launches, M&A
  • Analyst questions asking about specifics of the business
  • Executive answers with business substance (even if conversational in tone)
  • Market conditions, macro commentary with company-specific implications

EDGE CASES — apply these rules:
  • "Thank you, [name], and good afternoon" → BOILERPLATE (generic greeting even if it names someone)
  • "Great question" or "Thanks for asking" → BOILERPLATE (filler before a real answer)
  • Mixed sentences with both thanks AND substance → SUBSTANTIVE (substance wins)
  • Analyst intro that includes a question → SUBSTANTIVE (the question matters)
  • "We'll provide more detail at our Analyst Day" → BOILERPLATE (deflection, no info given)
  • Vague but directional ("We're optimistic about next quarter") → SUBSTANTIVE (guidance signal)
"""

JUDGE_PROMPTS = {
    "claude_haiku": {
        "provider": "anthropic",
        "model": "claude-haiku-4-5",
        "system": f"""You are a precise financial-text classifier.
{LABELING_RUBRIC}
Respond with ONLY the word BOILERPLATE or SUBSTANTIVE. Nothing else.""",
        "user_template": "Classify this sentence:\n\n\"{sentence}\""
    },

    "gpt4o_mini": {
        "provider": "openai",
        "model": "gpt-4o-mini",
        "system": f"""You are an expert NLP annotator for earnings-call transcripts.
{LABELING_RUBRIC}
Reply with exactly one word: BOILERPLATE or SUBSTANTIVE.""",
        "user_template": "Label the following sentence from an earnings call:\n\n\"{sentence}\""
    },

    "mistral_small": {
        "provider": "mistral",
        "model": "mistral-small-latest",
        "system": f"""You are a financial document analyst classifying earnings-call sentences.
{LABELING_RUBRIC}
Respond with exactly one word: BOILERPLATE or SUBSTANTIVE.""",
        "user_template": "Is this earnings-call sentence boilerplate or substantive?\n\n\"{sentence}\""
    },
}

print("Rubric and 3 judge prompts defined:")
for name, cfg in JUDGE_PROMPTS.items():
    print(f"  • {name:15s} ({cfg['provider']:9s} / {cfg['model']})")
print(f"\nRubric length: {len(LABELING_RUBRIC)} chars")

Rubric and 3 judge prompts defined:
  • claude_haiku    (anthropic / claude-haiku-4-5)
  • gpt4o_mini      (openai    / gpt-4o-mini)
  • mistral_small   (mistral   / mistral-small-latest)

Rubric length: 1915 chars


In [ ]:


# Cell 9: API client setup (Anthropic + OpenAI + Mistral via HTTP)

for pkg, import_name in [("anthropic", "anthropic"), ("openai", "openai"), ("requests", "requests")]:
    try:
        __import__(import_name)
    except ImportError:
        import subprocess, sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

import anthropic
import openai
import requests

# Uncomment and fill in if needed:
os.environ["ANTHROPIC_API_KEY"] = "YOUR_KEY_HERE"
os.environ["OPENAI_API_KEY"] = "YOUR_KEY_HERE"
os.environ["MISTRAL_API_KEY"] = "YOUR_KEY_HERE"

anthropic_key = os.environ.get("ANTHROPIC_API_KEY")
openai_key = os.environ.get("OPENAI_API_KEY")
mistral_key = os.environ.get("MISTRAL_API_KEY")

print(f"Anthropic API key: {'✓ found' if anthropic_key else '✗ missing'}")
print(f"OpenAI API key   : {'✓ found' if openai_key else '✗ missing'}")
print(f"Mistral API key  : {'✓ found' if mistral_key else '✗ missing'}")

if not (anthropic_key and openai_key and mistral_key):
    print("\n⚠️  One or more keys missing.")
else:
    anthropic_client = anthropic.Anthropic(api_key=anthropic_key)
    openai_client = openai.OpenAI(api_key=openai_key)
    # Mistral: just store the key, we'll call via requests
    MISTRAL_API_KEY = mistral_key
    print("\n✅ All three clients ready")


Anthropic API key: ✓ found
OpenAI API key   : ✓ found
Mistral API key  : ✓ found

✅ All three clients ready


In [10]:
# Cell 10a: Dry run on 10 sentences with all 3 judges

import time


def _normalize_label(text):
    if not text:
        return "UNKNOWN"
    t = text.strip().upper().replace("*", "").replace("`", "").replace('"', "").strip()
    first = t.split()[0] if t.split() else ""
    if first.startswith("BOILER"):
        return "BOILERPLATE"
    if first.startswith("SUBSTAN"):
        return "SUBSTANTIVE"
    if "BOILER" in t:
        return "BOILERPLATE"
    if "SUBSTAN" in t:
        return "SUBSTANTIVE"
    return "UNKNOWN"


def _call_anthropic(sentence, cfg):
    resp = anthropic_client.messages.create(
        model=cfg["model"],
        max_tokens=10,
        system=cfg["system"],
        messages=[{"role": "user", "content": cfg["user_template"].format(sentence=sentence)}],
    )
    return resp.content[0].text


def _call_openai(sentence, cfg):
    resp = openai_client.chat.completions.create(
        model=cfg["model"],
        max_tokens=10,
        temperature=0,
        messages=[
            {"role": "system", "content": cfg["system"]},
            {"role": "user", "content": cfg["user_template"].format(sentence=sentence)},
        ],
    )
    return resp.choices[0].message.content


def _call_mistral(sentence, cfg):
    """Direct HTTP call — bypasses the broken SDK."""
    response = requests.post(
        "https://api.mistral.ai/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {MISTRAL_API_KEY}",
            "Content-Type": "application/json",
        },
        json={
            "model": cfg["model"],
            "max_tokens": 10,
            "temperature": 0,
            "messages": [
                {"role": "system", "content": cfg["system"]},
                {"role": "user", "content": cfg["user_template"].format(sentence=sentence)},
            ],
        },
        timeout=30,
    )
    response.raise_for_status()
    return response.json()["choices"][0]["message"]["content"]


CALLERS = {
    "anthropic": _call_anthropic,
    "openai": _call_openai,
    "mistral": _call_mistral,
}

test_ids = []
test_ids.extend(df_gold_pool[df_gold_pool.speaker_role == "operator"].head(4)["sentence_id"].tolist())
test_ids.extend(df_gold_pool[(df_gold_pool.speaker_role == "executive") & (df_gold_pool.section == "presentation")].head(4)["sentence_id"].tolist())
test_ids.extend(df_gold_pool[df_gold_pool.speaker_role == "analyst"].head(2)["sentence_id"].tolist())

test_df = df_gold_pool[df_gold_pool.sentence_id.isin(test_ids)].reset_index(drop=True)
print(f"Testing on {len(test_df)} sentences\n")

results = []
for _, row in test_df.iterrows():
    print(f"--- sentence_id={row.sentence_id} | {row.speaker_role}/{row.section} ---")
    print(f"    {row.text[:120]}{'...' if len(row.text) > 120 else ''}")
    sentence_results = {"sentence_id": row.sentence_id, "text": row.text[:80]}
    for judge_name, cfg in JUDGE_PROMPTS.items():
        try:
            raw = CALLERS[cfg["provider"]](row.text, cfg)
            label = _normalize_label(raw)
            print(f"    {judge_name:15s} → {label:12s}  (raw: {raw!r})")
            sentence_results[judge_name] = label
        except Exception as e:
            print(f"    {judge_name:15s} → ERROR: {e}")
            sentence_results[judge_name] = "ERROR"
    results.append(sentence_results)
    print()

print("=" * 70)
print("Dry-run summary:")
df_test = pd.DataFrame(results)
print(df_test.to_string(index=False))

Testing on 10 sentences

--- sentence_id=935 | executive/presentation ---
    We expanded gross margin significantly and drove earnings growth while increasing investment in AI.
    claude_haiku    → SUBSTANTIVE   (raw: 'SUBSTANTIVE')
    gpt4o_mini      → SUBSTANTIVE   (raw: 'SUBSTANTIVE')
    mistral_small   → SUBSTANTIVE   (raw: 'SUBSTANTIVE')

--- sentence_id=20159 | executive/presentation ---
    Second, an enhanced LTL-specific pricing and invoicing system that drives faster speed to market, more intuitive contrac...
    claude_haiku    → SUBSTANTIVE   (raw: 'SUBSTANTIVE')
    gpt4o_mini      → SUBSTANTIVE   (raw: 'SUBSTANTIVE')
    mistral_small   → SUBSTANTIVE   (raw: 'SUBSTANTIVE')

--- sentence_id=21588 | executive/presentation ---
    Our Q3 performance demonstrates our team's strong commercial execution and our rigor in reducing structural cost through...
    claude_haiku    → SUBSTANTIVE   (raw: 'SUBSTANTIVE')
    gpt4o_mini      → SUBSTANTIVE   (raw: 'SUBSTANTIVE')
    mi

In [11]:
# Diagnostic: time a single Anthropic call vs 10 parallel calls

import time
from concurrent.futures import ThreadPoolExecutor

# Get 10 sample sentences
sample = df_gold_pool.head(10)["text"].tolist()
cfg = JUDGE_PROMPTS["claude_haiku"]

# Test 1: Serial (one at a time)
print("Test 1: Serial calls (10 sentences, one at a time)")
t0 = time.time()
for s in sample:
    _call_anthropic(s, cfg)
serial_time = time.time() - t0
print(f"  Serial: {serial_time:.1f}s ({10/serial_time:.2f} sent/sec)\n")

# Test 2: Parallel
print("Test 2: Parallel calls (10 sentences, 10 workers)")
t0 = time.time()
with ThreadPoolExecutor(max_workers=10) as pool:
    list(pool.map(lambda s: _call_anthropic(s, cfg), sample))
parallel_time = time.time() - t0
print(f"  Parallel: {parallel_time:.1f}s ({10/parallel_time:.2f} sent/sec)")

print(f"\nSpeedup: {serial_time / parallel_time:.1f}x")

Test 1: Serial calls (10 sentences, one at a time)
  Serial: 6.3s (1.58 sent/sec)

Test 2: Parallel calls (10 sentences, 10 workers)
  Parallel: 1.4s (7.15 sent/sec)

Speedup: 4.5x


In [12]:
# Cell 10: Full labeling — clean status output every 30 seconds

import time
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

JUDGE_LABELS_CACHE = CACHE_DIR / "judge_labels.parquet"

PROVIDER_CONCURRENCY = {"anthropic": 5, "openai": 10, "mistral": 5}


def _call_with_retry(caller, sentence, cfg, max_retries=5):
    for attempt in range(max_retries):
        try:
            return caller(sentence, cfg)
        except Exception as e:
            err_str = str(e).lower()
            if "rate" in err_str or "429" in err_str or "quota" in err_str or "too many" in err_str:
                time.sleep(min(60, 5 * (2 ** attempt)))
                continue
            if attempt == max_retries - 1:
                return None
            time.sleep(min(30, 2 ** attempt))
    return None


def _atomic_save(df, path):
    tmp = path.with_suffix(".tmp.parquet")
    df.to_parquet(tmp, index=False)
    tmp.replace(path)


if JUDGE_LABELS_CACHE.exists():
    df_judges = pd.read_parquet(JUDGE_LABELS_CACHE)
    print(f"Resuming. Existing columns: {list(df_judges.columns)}")
else:
    df_judges = df_gold_pool[["sentence_id", "text"]].copy()
    print("Starting fresh.")

save_lock = threading.Lock()
progress = {name: {"done": 0, "todo": 0} for name in JUDGE_PROMPTS}
progress_lock = threading.Lock()


def run_judge(judge_name, cfg):
    caller = CALLERS[cfg["provider"]]
    concurrency = PROVIDER_CONCURRENCY.get(cfg["provider"], 1)

    with save_lock:
        done = {}
        if judge_name in df_judges.columns:
            for _, row in df_judges.iterrows():
                v = row[judge_name]
                if pd.notna(v) and v != "UNKNOWN":
                    done[row["sentence_id"]] = v

    todo = [(sid, text) for sid, text in zip(df_gold_pool["sentence_id"], df_gold_pool["text"]) if sid not in done]
    if not todo:
        return done

    with progress_lock:
        progress[judge_name]["todo"] = len(todo)
        progress[judge_name]["done"] = 0

    def label_one(item):
        sid, text = item
        raw = _call_with_retry(caller, text, cfg)
        return sid, _normalize_label(raw) if raw else "UNKNOWN"

    results = dict(done)
    last_saved = len(done)

    with ThreadPoolExecutor(max_workers=concurrency) as pool:
        futures = [pool.submit(label_one, item) for item in todo]
        for fut in as_completed(futures):
            sid, label = fut.result()
            results[sid] = label
            with progress_lock:
                progress[judge_name]["done"] += 1
            if (len(results) - last_saved) >= 100:
                with save_lock:
                    df_judges[judge_name] = df_judges["sentence_id"].map(results)
                    _atomic_save(df_judges, JUDGE_LABELS_CACHE)
                last_saved = len(results)

    with save_lock:
        df_judges[judge_name] = df_judges["sentence_id"].map(results)
        _atomic_save(df_judges, JUDGE_LABELS_CACHE)

    return results


def status_reporter(stop_event):
    """Print a clean status update every 30 seconds."""
    t0 = time.time()
    while not stop_event.is_set():
        with progress_lock:
            snap = {k: dict(v) for k, v in progress.items()}
        elapsed = (time.time() - t0) / 60
        line = f"[{elapsed:5.1f} min] "
        for name, p in snap.items():
            if p["todo"] == 0:
                line += f"{name}: -- | "
            else:
                pct = 100 * p["done"] / p["todo"]
                rate = p["done"] / max(1, time.time() - t0)
                eta = (p["todo"] - p["done"]) / max(0.01, rate) / 60
                line += f"{name}: {p['done']}/{p['todo']} ({pct:.0f}%, ETA {eta:.0f}m) | "
        print(line, flush=True)
        stop_event.wait(30)


print("Starting all 3 judges concurrently...\n")
t_start = time.time()

stop_event = threading.Event()
reporter = threading.Thread(target=status_reporter, args=(stop_event,), daemon=True)
reporter.start()

with ThreadPoolExecutor(max_workers=3) as outer:
    futures = {outer.submit(run_judge, name, cfg): name for name, cfg in JUDGE_PROMPTS.items()}
    for fut in as_completed(futures):
        name = futures[fut]
        try:
            fut.result()
            print(f"  ✓ [{name}] complete")
        except Exception as e:
            print(f"  ✗ [{name}] failed: {e}")

stop_event.set()
elapsed = (time.time() - t_start) / 60
print(f"\n=== All judges complete in {elapsed:.1f} min ===\n")

print("=== Per-judge label distribution ===")
for j in JUDGE_PROMPTS:
    if j in df_judges.columns:
        print(f"\n{j}:")
        print(df_judges[j].value_counts().to_string())

Resuming. Existing columns: ['sentence_id', 'text', 'claude_haiku', 'gpt4o_mini', 'mistral_small', 'gold_label']
Starting all 3 judges concurrently...

[  0.0 min] claude_haiku: -- | gpt4o_mini: -- | mistral_small: -- | 
  ✓ [claude_haiku] complete
  ✓ [gpt4o_mini] complete
  ✓ [mistral_small] complete

=== All judges complete in 0.0 min ===

=== Per-judge label distribution ===

claude_haiku:
claude_haiku
SUBSTANTIVE    1981
BOILERPLATE     367

gpt4o_mini:
gpt4o_mini
SUBSTANTIVE    2117
BOILERPLATE     231

mistral_small:
mistral_small
SUBSTANTIVE    2164
BOILERPLATE     184


In [13]:
# Cell 11: Majority vote + inter-judge agreement

JUDGE_COLS = ["claude_haiku", "gpt4o_mini", "mistral_small"]

# Majority vote (gold label)
def majority_vote(row):
    labels = [row[c] for c in JUDGE_COLS if pd.notna(row[c]) and row[c] != "UNKNOWN"]
    if not labels:
        return "UNKNOWN"
    boil = labels.count("BOILERPLATE")
    sub = labels.count("SUBSTANTIVE")
    return "BOILERPLATE" if boil > sub else "SUBSTANTIVE"

df_judges["gold_label"] = df_judges.apply(majority_vote, axis=1)

# Agreement metrics
print("=== Inter-judge agreement ===")
total = len(df_judges)
unanimous = (df_judges[JUDGE_COLS].nunique(axis=1) == 1).sum()
print(f"Total sentences: {total:,}")
print(f"Unanimous (all 3 agree): {unanimous:,} ({100*unanimous/total:.1f}%)")
print(f"Disagreements (2-1 split): {total - unanimous:,} ({100*(total-unanimous)/total:.1f}%)")

# Pairwise agreement (Cohen's kappa)
from sklearn.metrics import cohen_kappa_score
print("\n=== Pairwise Cohen's kappa ===")
for i, j1 in enumerate(JUDGE_COLS):
    for j2 in JUDGE_COLS[i+1:]:
        kappa = cohen_kappa_score(df_judges[j1], df_judges[j2])
        agreement = (df_judges[j1] == df_judges[j2]).mean()
        print(f"  {j1:15s} ↔ {j2:15s} : κ = {kappa:.3f}  (raw agreement: {100*agreement:.1f}%)")

# Final gold distribution
print("\n=== Gold label distribution (majority vote) ===")
print(df_judges["gold_label"].value_counts())
print(f"\nClass balance: {100 * df_judges['gold_label'].value_counts(normalize=True).round(3)}")

# Save
df_judges.to_parquet(JUDGE_LABELS_CACHE, index=False)
print(f"\nSaved → {JUDGE_LABELS_CACHE}")

=== Inter-judge agreement ===
Total sentences: 2,348
Unanimous (all 3 agree): 2,140 (91.1%)
Disagreements (2-1 split): 208 (8.9%)

=== Pairwise Cohen's kappa ===
  claude_haiku    ↔ gpt4o_mini      : κ = 0.711  (raw agreement: 93.5%)
  claude_haiku    ↔ mistral_small   : κ = 0.613  (raw agreement: 91.9%)
  gpt4o_mini      ↔ mistral_small   : κ = 0.807  (raw agreement: 96.9%)

=== Gold label distribution (majority vote) ===
gold_label
SUBSTANTIVE    2116
BOILERPLATE     232
Name: count, dtype: int64

Class balance: gold_label
SUBSTANTIVE    90.1
BOILERPLATE     9.9
Name: proportion, dtype: float64

Saved → C:\Users\loren\Desktop\MTH9796\HW2\cache\judge_labels.parquet


In [14]:
# Cell 12: Audit the 208 disagreement cases

JUDGE_COLS = ["claude_haiku", "gpt4o_mini", "mistral_small"]

# Find disagreements
disagreements = df_judges[df_judges[JUDGE_COLS].nunique(axis=1) > 1].copy()
print(f"Total disagreements: {len(disagreements)}\n")

# Break disagreements down by pattern
disagreements["pattern"] = disagreements.apply(
    lambda r: "+".join(sorted(set(r[c][:4] for c in JUDGE_COLS))), axis=1
)
print("Disagreement patterns:")
print(disagreements["pattern"].value_counts())

# What does each judge "lean" on disagreement cases?
print("\nWhen judges disagree, who votes which way?")
for c in JUDGE_COLS:
    counts = disagreements[c].value_counts()
    print(f"  {c:15s}: BOILERPLATE={counts.get('BOILERPLATE',0):3d}  SUBSTANTIVE={counts.get('SUBSTANTIVE',0):3d}")

# Sample 25 disagreements stratified by section x speaker_role for human review
disagreements_with_meta = disagreements.merge(
    df_gold_pool[["sentence_id", "section", "speaker_role"]],
    on="sentence_id",
    how="left",
)
sample = disagreements_with_meta.groupby(
    ["section", "speaker_role"], group_keys=False
).apply(lambda g: g.sample(min(5, len(g)), random_state=RANDOM_SEED))

print(f"\n=== Sample of {len(sample)} disagreements for review ===\n")
for _, r in sample.iterrows():
    votes = " | ".join(f"{c.split('_')[0]}={r[c][:4]}" for c in JUDGE_COLS)
    print(f"[{r.speaker_role}/{r.section}] (gold={r.gold_label[:4]})  votes: {votes}")
    print(f"  → {r.text[:200]}")
    print()

Total disagreements: 208

Disagreement patterns:
pattern
BOIL+SUBS    208
Name: count, dtype: int64

When judges disagree, who votes which way?
  claude_haiku   : BOILERPLATE=196  SUBSTANTIVE= 12
  gpt4o_mini     : BOILERPLATE= 60  SUBSTANTIVE=148
  mistral_small  : BOILERPLATE= 13  SUBSTANTIVE=195

=== Sample of 17 disagreements for review ===

[executive/presentation] (gold=SUBS)  votes: claude=BOIL | gpt4o=SUBS | mistral=SUBS
  → In closing, I want to recognize our team for delivering another strong quarter while navigating a very difficult operating environment.

[executive/presentation] (gold=SUBS)  votes: claude=BOIL | gpt4o=SUBS | mistral=SUBS
  → And so you can have that discussion with your customer and then understand the market for that.

[executive/presentation] (gold=SUBS)  votes: claude=BOIL | gpt4o=SUBS | mistral=SUBS
  → And we just -- we are really, I think, as a company enjoying this phase.

[executive/presentation] (gold=SUBS)  votes: claude=BOIL | gpt4o=SUBS | mistr

In [15]:
# Cell 12b: Rule-based overrides for obvious gold-standard errors
# 
# Per the handout: "Correct the obvious ones; for genuinely ambiguous cases,
# document the rule you adopted." 
#
# We define explicit, defensible regex rules for the systematic disagreements
# we observed in Cell 12 (where Claude correctly flagged subtle boilerplate
# that GPT/Mistral missed). All other disagreements are left to the majority vote.

import re

# Force BOILERPLATE if these patterns match (with minimum confidence)
BOIL_OVERRIDE_PATTERNS = [
    # Operator handoff patterns (Mistral often votes SUBS on these incorrectly)
    (r"^(?:And\s+)?(?:the\s+)?[Nn]ext\s+question\s+(?:today\s+)?(?:is\s+)?(?:com(?:es|ing)|will\s+come)\s+from", "operator handoff"),
    (r"^Our\s+(?:first|next|final|last)\s+question\s+(?:is\s+)?(?:com(?:es|ing)|will\s+come)\s+from", "operator handoff"),
    
    # Analyst pleasantries — "congrats" / "congratulations" without a follow-up question in the same sentence
    (r"^(?:Hey,?\s+)?Congrat(?:ulation)?s\b(?!.*\?)", "analyst congrats"),
    
    # Generic thanks-for-question / appreciation
    (r"^(?:Thank\s+you|Thanks)(?:,\s+\w+)?[,.\s]+(?:I\s+(?:very\s+much\s+)?appreciate|that('?s)?\s+(?:very\s+)?helpful)", "thanks/appreciation"),
    (r"^I\s+(?:very\s+much\s+)?appreciate\s+(?:the|your)\s+(?:comments?|color|context|insight)", "appreciation filler"),
    
    # Pure-color CEO platitudes (no numbers, no business content)
    (r"^You\s+can\s+feel\s+the\s+energy\b", "CEO color"),
    (r"^I\s+want\s+to\s+(?:thank|recognize)\s+(?:our|the)\s+(?:team|employees|associates)", "team thanks"),
    
    # Analyst preambles before the actual question
    (r"^That'?s\s+kind\s+of\s+what\s+I\s+(?:would|wanted?)\s+(?:like\s+to\s+)?(?:just\s+)?discuss", "analyst preamble"),
    (r"^(?:Maybe\s+just\s+)?to\s+ask\s+about\b.*,\s+another\s+angle", "analyst preamble"),
    
    # Closing remarks / signoffs
    (r"^In\s+closing,?\s+I\s+want\s+to", "closing remarks"),
    (r"^With\s+that,?\s+(?:I'?ll|let\s+me|we'?ll)\s+(?:turn|hand|open)", "call handoff"),
    
    # Pure deflections
    (r"^(?:For\s+competitive\s+reasons|At\s+this\s+time|At\s+this\s+point),\s+(?:we|I)(?:'re|'ll|\s+won'?t|\s+can'?t|\s+do\s+not)\s+(?:going\s+to\s+)?(?:comment|discuss|share|provide|disclose)", "deflection"),
]

# Force SUBSTANTIVE if these patterns match (rare, but for defensible cases)
SUBS_OVERRIDE_PATTERNS = [
    # Specific dollar/percentage figures with business context
    # (we don't define these — majority vote already handles these well)
]


def apply_overrides(row):
    """Return (new_label, reason) or (original_label, None) if no rule applies."""
    text = row["text"]
    original = row["gold_label"]
    
    # Only override on the disagreement subset (where judges split 2-1)
    judge_vals = [row[c] for c in JUDGE_COLS]
    if len(set(judge_vals)) == 1:
        return original, None  # all unanimous — leave alone
    
    for pattern, reason in BOIL_OVERRIDE_PATTERNS:
        if re.search(pattern, text):
            return "BOILERPLATE", reason
    for pattern, reason in SUBS_OVERRIDE_PATTERNS:
        if re.search(pattern, text):
            return "SUBSTANTIVE", reason
    return original, None


# Apply
overrides_applied = []
for idx, row in df_judges.iterrows():
    new_label, reason = apply_overrides(row)
    if reason is not None and new_label != row["gold_label"]:
        overrides_applied.append({
            "sentence_id": row["sentence_id"],
            "old_label": row["gold_label"],
            "new_label": new_label,
            "reason": reason,
            "text": row["text"],
        })
        df_judges.at[idx, "gold_label"] = new_label

print(f"=== Override summary ===")
print(f"Total overrides applied: {len(overrides_applied)}")

if overrides_applied:
    df_overrides = pd.DataFrame(overrides_applied)
    print(f"\nOverrides by reason:")
    print(df_overrides["reason"].value_counts().to_string())
    
    print(f"\nDirection of overrides:")
    print(df_overrides.apply(lambda r: f"{r.old_label[:4]} → {r.new_label[:4]}", axis=1).value_counts().to_string())
    
    print(f"\n=== First 15 overrides ===")
    for _, r in df_overrides.head(15).iterrows():
        print(f"  [{r.reason}] {r.old_label[:4]} → {r.new_label[:4]}: {r.text[:140]}")

# New gold distribution
print(f"\n=== Gold label distribution AFTER overrides ===")
print(df_judges["gold_label"].value_counts())
print(f"\nNew class balance: {(df_judges['gold_label'].value_counts(normalize=True) * 100).round(1).to_dict()}")

# Save
df_judges.to_parquet(JUDGE_LABELS_CACHE, index=False)
print(f"\n✅ Saved → {JUDGE_LABELS_CACHE}")

=== Override summary ===
Total overrides applied: 4

Overrides by reason:
reason
CEO color           1
closing remarks     1
analyst preamble    1
analyst congrats    1

Direction of overrides:
SUBS → BOIL    4

=== First 15 overrides ===
  [CEO color] SUBS → BOIL: You can feel the energy and the enthusiasm walking around campus.
  [closing remarks] SUBS → BOIL: In closing, I want to recognize our team for delivering another strong quarter while navigating a very difficult operating environment.
  [analyst preamble] SUBS → BOIL: Maybe just to ask about the infrastructure, another angle of this.
  [analyst congrats] SUBS → BOIL: Congrats on the Intra-Cellular deal last week.

=== Gold label distribution AFTER overrides ===
gold_label
SUBSTANTIVE    2112
BOILERPLATE     236
Name: count, dtype: int64

New class balance: {'SUBSTANTIVE': 89.9, 'BOILERPLATE': 10.1}

✅ Saved → C:\Users\loren\Desktop\MTH9796\HW2\cache\judge_labels.parquet


In [16]:
# Cell 12c: Inspect ALL 208 disagreements by pattern to find override candidates

JUDGE_COLS = ["claude_haiku", "gpt4o_mini", "mistral_small"]

disagreements = df_judges[df_judges[JUDGE_COLS].nunique(axis=1) > 1].copy()
disagreements = disagreements.merge(
    df_gold_pool[["sentence_id", "section", "speaker_role"]],
    on="sentence_id",
    how="left",
)

# Build vote-pattern label
def vote_pattern(row):
    return f"C={row.claude_haiku[:4]}/G={row.gpt4o_mini[:4]}/M={row.mistral_small[:4]}"
disagreements["votes"] = disagreements.apply(vote_pattern, axis=1)

print(f"Total disagreements: {len(disagreements)}\n")
print("Vote patterns:")
print(disagreements["votes"].value_counts())
print()

# Print ALL 208 grouped by pattern, showing speaker_role for context
print("=" * 100)
for pattern, group in disagreements.groupby("votes"):
    print(f"\n### {pattern}  —  {len(group)} sentences  (current gold={group['gold_label'].iloc[0][:4]})")
    print("-" * 100)
    for _, r in group.iterrows():
        print(f"  [{r.speaker_role:9s}/{r.section:12s}] {r.text[:160]}")

Total disagreements: 208

Vote patterns:
votes
C=BOIL/G=SUBS/M=SUBS    135
C=BOIL/G=BOIL/M=SUBS     52
C=BOIL/G=SUBS/M=BOIL      9
C=SUBS/G=BOIL/M=SUBS      8
C=SUBS/G=SUBS/M=BOIL      4
Name: count, dtype: int64


### C=BOIL/G=BOIL/M=SUBS  —  52 sentences  (current gold=BOIL)
----------------------------------------------------------------------------------------------------
  [executive/qa          ] And finally, a big thank you to Team FedEx for your outstanding work in Q2 and throughout this peak season with just one more week to go.
  [executive/qa          ] And again, in terms of NIKE Direct digital commerce.
  [analyst  /qa          ] And then next question is really sort of a follow-on from Betsy's line of questioning.
  [analyst  /qa          ] Maybe we could just ask you to unpack a little bit.
  [analyst  /qa          ] I wanted to follow back -- follow up on a comment.
  [operator /qa          ] The next question is from Betsy Graseck with Morgan Stanley.
  [operator /qa  

In [17]:
# Cell 12d: Additional narrow override rules

import re

# Additional patterns we identified after seeing all 208 disagreements
ADDITIONAL_BOIL_PATTERNS = [
    # Pure summary/intro openings with no figures
    (r"^In\s+summary,?\s+it\s+was\s+a", "summary opener"),
    (r"^It'?s\s+my\s+pleasure\s+to\s+present", "presentation opener"),
    (r"^Welcome\s+to\s+(?:our|the)\s+(?:company('?s)?|firm('?s)?)\s+review", "welcome opener"),
    (r"^Let\s+me\s+begin\s+by\s+noting", "speech opener"),
    (r"^Before\s+I\s+(?:get\s+into|turn\s+to)\s+the\s+numbers", "speech opener"),
    
    # Personal anecdotes about colleagues / new hires intros
    (r"^(?:He|She)'?s\s+a\s+\d+(?:-year)?\s+(?:NIKE|company|firm|veteran)", "colleague intro"),
    (r"^[A-Z][a-z]+\s+(?:was\s+my\s+first\s+hire|is\s+a\s+world-class|is\s+naturally\s+talented|has\s+been\s+chasing)", "colleague praise"),
    (r"^(?:Ray|Doug|Bob|Bill|Jeff|Rory)\s+(?:was|is|has)\s+", "named colleague reference"),
    
    # "I want to congratulate / thank our team" type recognitions
    (r"^I\s+want(?:ed)?\s+to\s+(?:congratulate|thank|recognize)\s+(?:our|the|my)\s+(?:team|colleagues|associates|employees|Innovative|FedEx)", "team recognition"),
    (r"^A\s+big\s+thank\s+you\s+to\s+(?:Team|our|the)", "team thanks"),
    (r"^I'?(?:m|d\s+like\s+to)\s+(?:also\s+)?(?:very\s+)?grateful\s+for\s+(?:the\s+)?dedication", "gratitude"),
    
    # Generic encouragement / positive feeling
    (r"^I'?m\s+excited\s+about\s+the\s+momentum", "vague excitement"),
    (r"^At\s+BlackRock,?\s+we\s+are\s+energized\s+by", "company spirit"),
    
    # Upcoming-event announcements (no info given, just "we'll be at X")
    (r"^We\s+will\s+be\s+at\s+the\b", "upcoming event"),
    (r"^Please\s+join\s+us\s+at\b", "event invitation"),
    (r"^Let\s+me\s+highlight\s+an\s+upcoming\s+event", "upcoming event"),
    
    # Reading transitions ("Now let me turn to...", "Turning to...")
    (r"^Now\s+let\s+me\s+turn\s+to\s+our\s+(?:first|second|third|fourth)\s+quarter", "section transition"),
    (r"^Now\s+I'?d\s+like\s+to\s+turn\s+the\s+call\s+over\s+to", "call handoff"),
    (r"^I'?ll\s+then\s+turn\s+the\s+call\s+over\s+to", "call handoff"),
    
    # Self-deprecation / filler
    (r"^Now\s+I'?m\s+going\s+to\s+stop\s+talking", "self-filler"),
    (r"^That'?s\s+the\s+reason\s+why\s+I\s+jumped\s+in", "self-filler"),
    
    # "I encourage you to review our 10-Q" type pointers
    (r"^I\s+encourage\s+you\s+to\s+review\s+our\s+(?:upcoming\s+)?10-(?:Q|K)", "filing pointer"),
]


def apply_additional_overrides(row):
    """Same as before but with the new pattern set, and apply to ANY disagreement (not just specific patterns)."""
    text = row["text"]
    judge_vals = [row[c] for c in JUDGE_COLS]
    if len(set(judge_vals)) == 1:
        return row["gold_label"], None
    for pattern, reason in ADDITIONAL_BOIL_PATTERNS:
        if re.search(pattern, text):
            return "BOILERPLATE", reason
    return row["gold_label"], None


# Apply
new_overrides = []
for idx, row in df_judges.iterrows():
    new_label, reason = apply_additional_overrides(row)
    if reason is not None and new_label != row["gold_label"]:
        new_overrides.append({
            "sentence_id": row["sentence_id"],
            "old_label": row["gold_label"],
            "new_label": new_label,
            "reason": reason,
            "text": row["text"],
        })
        df_judges.at[idx, "gold_label"] = new_label

print(f"=== Additional overrides ===")
print(f"Total: {len(new_overrides)}")

if new_overrides:
    df_new = pd.DataFrame(new_overrides)
    print(f"\nBy reason:")
    print(df_new["reason"].value_counts().to_string())
    print(f"\n=== All new overrides ===")
    for _, r in df_new.iterrows():
        print(f"  [{r.reason}] {r.old_label[:4]} → {r.new_label[:4]}: {r.text[:160]}")

# Final gold distribution
print(f"\n=== FINAL gold label distribution ===")
print(df_judges["gold_label"].value_counts())
print(f"\nFinal class balance: {(df_judges['gold_label'].value_counts(normalize=True) * 100).round(1).to_dict()}")

df_judges.to_parquet(JUDGE_LABELS_CACHE, index=False)
print(f"\n✅ Saved → {JUDGE_LABELS_CACHE}")

=== Additional overrides ===
Total: 11

By reason:
reason
speech opener       2
colleague praise    2
colleague intro     1
summary opener      1
company spirit      1
event invitation    1
team recognition    1
vague excitement    1
filing pointer      1

=== All new overrides ===
  [speech opener] SUBS → BOIL: Before I get into the numbers, I'd like to provide some qualitative business highlights from the quarter.
  [colleague intro] SUBS → BOIL: He's a 25-year NIKE veteran, has deep and broad product and marketplace experience and he's a tremendous leader.
  [summary opener] SUBS → BOIL: In summary, it was a solid start to the year.
  [colleague praise] SUBS → BOIL: Ray was my first hire when I joined Wells, and I worked with Ray for much of the 38 years we've known each other.
  [colleague praise] SUBS → BOIL: Doug is a world-class banker, and he's working alongside the great team we've assembled to continue to grow the franchise.
  [speech opener] SUBS → BOIL: Let me begin by noti

In [18]:
# Cell 12e: Manual sentence-level overrides for the contested disagreements.
#
# Per the handout: "Correct the obvious ones; for genuinely ambiguous cases,
# document the rule you adopted." 
#
# We hand-reviewed all 208 disagreements after the rule-based passes and
# manually flagged additional cases where the majority vote was clearly wrong.
# Each override is enumerated below with its reason. Cases that the rubric's
# "vague but directional → SUBSTANTIVE" rule covers were left alone.

MANUAL_OVERRIDES = [
    # === SUBS → BOIL: pure color / corporate platitudes (Claude was right) ===
    ("And we just -- we are really, I think, as a company enjoying this phase.", "BOILERPLATE", "color, no info"),
    ("You have to dare to try, and I deeply admire the monumental effort.", "BOILERPLATE", "color, no info"),
    ("Above all, they remind us of the hard work and the hustle that is required to win.", "BOILERPLATE", "color, no info"),
    ("Patient safety is always an absolute priority for us.", "BOILERPLATE", "corporate platitude"),
    ("We're in the business of delivering magically -- projects that are magical on the front line.", "BOILERPLATE", "corporate platitude"),
    ("We continue to deliver incredible growth.", "BOILERPLATE", "vague boast, no number"),
    ("We're very excited about the plans going forward.", "BOILERPLATE", "filler enthusiasm"),
    ("It's been a very successful program for us so far.", "BOILERPLATE", "vague boast"),
    ("It's really important to bring people together.", "BOILERPLATE", "platitude"),
    ("We want to be consistently the world's best.", "BOILERPLATE", "aspirational platitude"),
    ("Our first priority and will always be safety above all.", "BOILERPLATE", "corporate platitude"),
    ("Because at the end of the day, if our customers are successful, we win.", "BOILERPLATE", "platitude"),
    ("And every single Palantirian is special.", "BOILERPLATE", "color"),
    
    # === SUBS → BOIL: personal anecdotes about colleagues ===
    ("And one thing I've observed over the years is there was not a day that Bob Kierlin does not read the Wall Street Journal from cover to cover.", "BOILERPLATE", "personal anecdote"),
    ("And I really asked them to get to know Jeff Watts better.", "BOILERPLATE", "personal anecdote"),
    ("For those of you that don't know, Rory has been chasing a Masters victory for 14 years.", "BOILERPLATE", "personal anecdote"),
    ("I know her expertise and insights will be invaluable as we continue to transform FedEx.", "BOILERPLATE", "colleague praise"),
    
    # === SUBS → BOIL: analyst preambles / fillers ===
    ("Real quickly, on the server side of the business, kind of off the prior question.", "BOILERPLATE", "analyst preamble"),
    ("So this is just a context of why I'm asking this question on expenses.", "BOILERPLATE", "analyst preamble"),
    ("A lot of my questions have been answered, but let me ask one for John here.", "BOILERPLATE", "analyst preamble"),
    
    # === SUBS → BOIL: section transitions in presentation ===
    ("Now turning to our consolidated statement of earnings for the second quarter of 2025.", "BOILERPLATE", "section transition"),
    ("Let me turn to the outlook for the second quarter.", "BOILERPLATE", "section transition"),
    ("Let me turn to the outlook for the first quarter.", "BOILERPLATE", "section transition"),
    ("Moving to our Professional Visualization business.", "BOILERPLATE", "section transition"),
    ("Turning to the cash flow and capital allocation slide.", "BOILERPLATE", "section transition"),
    
    # === SUBS → BOIL: events / filings pointers ===
    ("Please join us at CES in Las Vegas, where Jensen will deliver a keynote on January", "BOILERPLATE", "event invitation"),
    ("And next week will be Nike's 10th Air Max Day.", "BOILERPLATE", "event mention"),
    
    # === SUBS → BOIL: the rest ===
    ("That's good for consumers, that's good for our partners, and that's good for NIKE.", "BOILERPLATE", "platitude"),
    ("And our challenge to our team is we have to keep moving that forward.", "BOILERPLATE", "platitude"),
    ("It's a fairly thankless effort, to be quite honest.", "BOILERPLATE", "color"),
]

# Apply overrides by exact text match
override_count = 0
override_log = []
for text_match, new_label, reason in MANUAL_OVERRIDES:
    matches = df_judges[df_judges["text"] == text_match]
    if len(matches) == 0:
        print(f"⚠️  NO MATCH: {text_match[:80]}")
        continue
    elif len(matches) > 1:
        print(f"⚠️  MULTIPLE MATCHES ({len(matches)}): {text_match[:80]}")
    
    idx = matches.index[0]
    old_label = df_judges.at[idx, "gold_label"]
    if old_label != new_label:
        df_judges.at[idx, "gold_label"] = new_label
        override_count += 1
        override_log.append({
            "old": old_label[:4], "new": new_label[:4],
            "reason": reason, "text": text_match[:120]
        })

print(f"\n=== Manual overrides applied: {override_count} ===\n")
for log in override_log:
    print(f"  [{log['reason']:25s}] {log['old']} → {log['new']}: {log['text']}")

# Final gold distribution
print(f"\n=== FINAL gold label distribution ===")
print(df_judges["gold_label"].value_counts())
print(f"\nFinal class balance: {(df_judges['gold_label'].value_counts(normalize=True) * 100).round(1).to_dict()}")

# Save
df_judges.to_parquet(JUDGE_LABELS_CACHE, index=False)
print(f"\n✅ Saved → {JUDGE_LABELS_CACHE}")


=== Manual overrides applied: 25 ===

  [color, no info           ] SUBS → BOIL: And we just -- we are really, I think, as a company enjoying this phase.
  [color, no info           ] SUBS → BOIL: Above all, they remind us of the hard work and the hustle that is required to win.
  [corporate platitude      ] SUBS → BOIL: Patient safety is always an absolute priority for us.
  [corporate platitude      ] SUBS → BOIL: We're in the business of delivering magically -- projects that are magical on the front line.
  [vague boast, no number   ] SUBS → BOIL: We continue to deliver incredible growth.
  [filler enthusiasm        ] SUBS → BOIL: We're very excited about the plans going forward.
  [vague boast              ] SUBS → BOIL: It's been a very successful program for us so far.
  [platitude                ] SUBS → BOIL: It's really important to bring people together.
  [aspirational platitude   ] SUBS → BOIL: We want to be consistently the world's best.
  [corporate platitude      ] SUBS

In [19]:
# Cell 13: Stratified 60/20/20 split — TEST IS FROZEN AFTER THIS

from sklearn.model_selection import train_test_split

SPLIT_CACHE = CACHE_DIR / "splits.parquet"

# Build the labeled dataset
df_labeled = df_gold_pool.merge(
    df_judges[["sentence_id", "gold_label"] + JUDGE_COLS],
    on="sentence_id",
    how="inner",
)

df_labeled = df_labeled[df_labeled["gold_label"] != "UNKNOWN"].reset_index(drop=True)
print(f"Labeled dataset: {len(df_labeled):,} sentences")
print(f"Class balance: {df_labeled['gold_label'].value_counts(normalize=True).round(3).to_dict()}")

y = df_labeled["gold_label"].values

# 80% (train+val) / 20% test
trainval_idx, test_idx = train_test_split(
    range(len(df_labeled)),
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_SEED,
)

# 75/25 split of the 80% → train (60% of total) and val (20% of total)
trainval_y = y[trainval_idx]
train_idx, val_idx = train_test_split(
    trainval_idx,
    test_size=0.25,
    stratify=trainval_y,
    random_state=RANDOM_SEED,
)

df_labeled["split"] = "test"
df_labeled.loc[train_idx, "split"] = "train"
df_labeled.loc[val_idx, "split"] = "val"
df_labeled.loc[test_idx, "split"] = "test"

df_labeled.to_parquet(SPLIT_CACHE, index=False)

print(f"\n=== Split sizes ===")
print(df_labeled["split"].value_counts())
print(f"\n=== Class distribution per split ===")
print(pd.crosstab(df_labeled["split"], df_labeled["gold_label"], margins=True))
print(f"\n=== Boilerplate proportion per split ===")
print(df_labeled.groupby("split")["gold_label"].apply(lambda s: (s == "BOILERPLATE").mean().round(3)))

print(f"\n✅ Splits frozen and saved → {SPLIT_CACHE}")
print(f"⚠️  TEST SET FROZEN — do not look at it until final evaluation.")

Labeled dataset: 2,348 sentences
Class balance: {'SUBSTANTIVE': 0.884, 'BOILERPLATE': 0.116}

=== Split sizes ===
split
train    1408
val       470
test      470
Name: count, dtype: int64

=== Class distribution per split ===
gold_label  BOILERPLATE  SUBSTANTIVE   All
split                                     
test                 54          416   470
train               163         1245  1408
val                  55          415   470
All                 272         2076  2348

=== Boilerplate proportion per split ===
split
test     0.115
train    0.116
val      0.117
Name: gold_label, dtype: float64

✅ Splits frozen and saved → C:\Users\loren\Desktop\MTH9796\HW2\cache\splits.parquet
⚠️  TEST SET FROZEN — do not look at it until final evaluation.


In [20]:
# Cell 14: Hand-crafted regex / numeric features
#
# These are domain-specific signals tailored to earnings-call structure.
# Each feature is a binary 0/1 flag or a count, designed to catch obvious
# boilerplate or substantive patterns that a frozen embedding might miss.

import re
import numpy as np

# --- Pattern definitions ---

# Boilerplate signals
PATTERNS_BOIL = {
    "f_operator_intro": re.compile(
        r"(?i)\b(?:next\s+question|first\s+question|final\s+question|last\s+question)\b.*\b(?:com(?:es|ing)|will\s+come|is\s+from)\b"
    ),
    "f_speaker_intro_with_firm": re.compile(
        r"\b(?:Bernstein|Goldman|Morgan\s+Stanley|JPMorgan|JP\s+Morgan|Wells\s+Fargo|Citi(?:group)?|Bank\s+of\s+America|BofA|"
        r"Wolfe|Evercore|Jefferies|Cantor\s+Fitzgerald|Deutsche\s+Bank|UBS|Barclays|Cowen|Citigroup|Piper\s+Sandler|"
        r"Stifel|Raymond\s+James|William\s+Blair|Truist|Baird|Mizuho|Nomura|HSBC|Macquarie|Seaport|RBC|BMO|"
        r"Oppenheimer|Needham|Wedbush|Bernstein\s+Research|Seaport\s+Global)\b"
    ),
    "f_operator_instructions": re.compile(r"(?i)\[Operator\s+Instructions\]"),
    "f_safe_harbor": re.compile(
        r"(?i)\b(?:forward[- ]looking\s+statements?|safe\s+harbor|risk\s+factors|10[- ]K|10[- ]Q|"
        r"private\s+securities\s+litigation|reform\s+act|materially\s+different|actual\s+results)\b"
    ),
    "f_replay_recording": re.compile(
        r"(?i)\b(?:this\s+(?:call|webcast)\s+is\s+being\s+recorded|replay\s+(?:will\s+be|is)\s+available|webcast\s+replay|"
        r"available\s+on\s+our\s+(?:investor\s+relations\s+)?website)\b"
    ),
    "f_thanks_only": re.compile(
        r"^\s*(?:Thank\s+you|Thanks)(?:[\s,.]+(?:very\s+much|all|everyone|operator|\w+))?\s*[.!?]?\s*$"
    ),
    "f_greeting": re.compile(
        r"(?i)\b(?:good\s+(?:morning|afternoon|evening)|welcome\s+to|hello\s+(?:everyone|all)|"
        r"thank\s+you\s+for\s+(?:joining|standing\s+by))\b"
    ),
    "f_call_handoff": re.compile(
        r"(?i)\b(?:turn\s+(?:the\s+call|it)\s+(?:over\s+)?(?:to|back)|hand\s+(?:the\s+call|it)\s+(?:over\s+)?(?:to|back)|"
        r"with\s+that,?\s+(?:I'?ll|let\s+me|we'?ll)\s+(?:turn|hand|open|move))\b"
    ),
    "f_appreciate_question": re.compile(
        r"(?i)^(?:(?:great|good|thanks?\s+for\s+the)\s+question|appreciate\s+(?:the|your)\s+question|"
        r"thanks?\s+(?:for\s+)?(?:taking|asking)\s+(?:my|the)\s+question)"
    ),
    "f_congrats": re.compile(
        r"(?i)^(?:Hey,?\s+)?Congrat(?:ulation)?s\b"
    ),
    "f_section_transition": re.compile(
        r"(?i)^(?:Now\s+(?:let\s+me\s+)?turn(?:ing)?(?:\s+to)?|"
        r"(?:Now\s+)?Turning\s+to|"
        r"(?:Now\s+)?Moving\s+(?:on\s+)?to|"
        r"Let\s+me\s+turn\s+to|"
        r"With\s+that,?)"
    ),
    "f_closing_remarks": re.compile(
        r"(?i)\b(?:in\s+closing|in\s+conclusion|to\s+(?:wrap|sum)\s+(?:this\s+)?up|"
        r"with\s+that,?\s+I'?ll\s+open\s+(?:it\s+up|the\s+call|the\s+line)\s+(?:for|to)\s+questions)\b"
    ),
}

# Substantive signals
PATTERNS_SUBS = {
    "f_dollar_amount": re.compile(r"\$\s*\d"),
    "f_percent": re.compile(r"\d+(?:\.\d+)?\s*%"),
    "f_basis_points": re.compile(r"(?i)\b\d+\s*(?:basis\s+points?|bps|bp)\b"),
    "f_quarter_ref": re.compile(r"(?i)\bQ[1-4]\b|\b(?:first|second|third|fourth)\s+quarter\b|\bfiscal\s+(?:year\s+)?\d{4}\b"),
    "f_year_over_year": re.compile(r"(?i)\b(?:year[- ]over[- ]year|y/y|yoy|year[- ]on[- ]year)\b"),
    "f_growth_decline": re.compile(
        r"(?i)\b(?:grew|grow(?:th|ing)?|declin(?:ed|ing|e)|increased?|decreased?|rose|fell|dropped|surged|jumped)\b"
    ),
    "f_segment_term": re.compile(
        r"(?i)\b(?:segment|division|business\s+unit|product\s+line|geography|region)\b"
    ),
    "f_guidance_term": re.compile(
        r"(?i)\b(?:guid(?:ance|ing)|outlook|forecast|expect(?:ation)?s?|anticipate|projected?|target(?:ing|s|ed)?)\b"
    ),
    "f_margin_term": re.compile(
        r"(?i)\b(?:gross\s+margin|operating\s+margin|net\s+margin|EBITDA|operating\s+income|net\s+income|EPS|"
        r"earnings\s+per\s+share|free\s+cash\s+flow|FCF|return\s+on\s+(?:equity|assets|invested\s+capital))\b"
    ),
}


def extract_features(text):
    """Compute all regex + numeric features for one sentence."""
    feats = {}
    for name, pat in PATTERNS_BOIL.items():
        feats[name] = int(bool(pat.search(text)))
    for name, pat in PATTERNS_SUBS.items():
        feats[name] = int(bool(pat.search(text)))
    
    # Numeric / structural features
    feats["f_char_len"] = len(text)
    feats["f_word_count"] = len(text.split())
    feats["f_digit_count"] = sum(c.isdigit() for c in text)
    feats["f_dollar_count"] = text.count("$")
    feats["f_question_mark"] = int("?" in text)
    feats["f_starts_capital"] = int(text[0].isupper() if text else 0)
    feats["f_uppercase_ratio"] = sum(c.isupper() for c in text) / max(1, len(text))
    feats["f_punct_ratio"] = sum(1 for c in text if c in ".,;:!?") / max(1, len(text))
    
    return feats


# Apply to all labeled sentences
print("Extracting regex features...")
df_split = pd.read_parquet(SPLIT_CACHE)
feature_dicts = [extract_features(t) for t in df_split["text"]]
feature_names = list(feature_dicts[0].keys())
df_features = pd.DataFrame(feature_dicts, columns=feature_names)

print(f"Built {len(feature_names)} regex/numeric features over {len(df_features)} sentences\n")

# Show feature firing rates
print("=== Feature firing rates (overall) ===")
binary_feats = [c for c in feature_names if c.startswith("f_") and df_features[c].max() <= 1 and df_features[c].dtype != float]
firing = df_features[binary_feats].mean().sort_values(ascending=False)
print(firing.to_string())

# Quick correlation check: which features fire more on BOILERPLATE?
print("\n=== Top discriminating features (BOIL rate vs SUBS rate) ===")
df_features_with_label = df_features.copy()
df_features_with_label["gold_label"] = df_split["gold_label"].values

binary_feats_only = [c for c in binary_feats]
discrim = []
for c in binary_feats_only:
    boil_rate = df_features_with_label[df_features_with_label["gold_label"] == "BOILERPLATE"][c].mean()
    sub_rate = df_features_with_label[df_features_with_label["gold_label"] == "SUBSTANTIVE"][c].mean()
    discrim.append((c, boil_rate, sub_rate, boil_rate - sub_rate))
discrim_df = pd.DataFrame(discrim, columns=["feature", "boil_rate", "sub_rate", "diff"])
discrim_df = discrim_df.reindex(discrim_df["diff"].abs().sort_values(ascending=False).index).head(15)
print(discrim_df.to_string(index=False))

# Save
FEATURES_CACHE = CACHE_DIR / "features_regex.parquet"
df_features.insert(0, "sentence_id", df_split["sentence_id"].values)
df_features.to_parquet(FEATURES_CACHE, index=False)
print(f"\n✅ Saved → {FEATURES_CACHE}")


# Note: 5 features fire on < 0.5% of train+val sentences and contribute essentially
# no signal: f_replay_recording (0 firings), f_closing_remarks, f_congrats,
# f_appreciate_question (1 firing each), f_safe_harbor (5 firings).
# We retain them in the feature matrix for completeness and for transparent
# reporting in the write-up. The winning classifier (logreg_emb) does not use
# the regex features at all, so this has no effect on the deployed model.

Extracting regex features...
Built 29 regex/numeric features over 2348 sentences

=== Feature firing rates (overall) ===
f_starts_capital             0.988075
f_growth_decline             0.128620
f_percent                    0.098807
f_dollar_amount              0.086882
f_guidance_term              0.083475
f_quarter_ref                0.080920
f_question_mark              0.057496
f_year_over_year             0.030664
f_speaker_intro_with_firm    0.024702
f_margin_term                0.022998
f_operator_intro             0.015758
f_greeting                   0.011073
f_call_handoff               0.007240
f_basis_points               0.006814
f_segment_term               0.006388
f_section_transition         0.005963
f_operator_instructions      0.005111
f_safe_harbor                0.002981
f_congrats                   0.000852
f_closing_remarks            0.000426
f_replay_recording           0.000426
f_thanks_only                0.000000
f_appreciate_question        0.000000

=== 

In [21]:
# Diagnostic: directly search the dataset
import re
test_pat = re.compile(r"(?i)(?:great|good)\s+question|appreciate\s+(?:the|your)\s+question|thanks?\s+(?:for\s+)?(?:taking|asking)\s+(?:my|the|that)\s+question")

matches = df_split[df_split["text"].str.contains(test_pat)]
print(f"Direct regex match: {len(matches)} sentences")
if len(matches) > 0:
    for t in matches["text"].head(5):
        print(f"  - {t[:150]}")

# Also check raw "great question" / "thanks for the question" etc
print(f"\nSimple substring counts:")
for phrase in ["great question", "Great question", "thanks for the question", 
               "Thanks for the question", "appreciate the question", 
               "Thanks for taking", "thanks for taking"]:
    n = df_split["text"].str.contains(phrase, case=False, regex=False).sum()
    print(f"  {phrase!r:35s} : {n}")

Direct regex match: 1 sentences
  - It's different than what I've said about our journey around the consumer platforms in the business, but I appreciate the question.

Simple substring counts:
  'great question'                    : 0
  'Great question'                    : 0
  'thanks for the question'           : 1
  'Thanks for the question'           : 1
  'appreciate the question'           : 1
  'Thanks for taking'                 : 0
  'thanks for taking'                 : 0


In [22]:
# Cell 15: Generate sentence embeddings
#
# We use sentence-transformers locally — no API costs, no rate limits.
# all-MiniLM-L6-v2 is the standard fast/good sentence embedding model:
# 384-dim, ~80MB download, ~50 sentences/sec on CPU.
#
# Cached to disk so we never re-encode.

EMBEDDINGS_CACHE = CACHE_DIR / "embeddings.npy"
EMBEDDINGS_IDS_CACHE = CACHE_DIR / "embeddings_ids.parquet"

# Install if needed
try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "sentence-transformers", "-q"])
    from sentence_transformers import SentenceTransformer

import numpy as np

if EMBEDDINGS_CACHE.exists() and EMBEDDINGS_IDS_CACHE.exists():
    print(f"Loading cached embeddings from {EMBEDDINGS_CACHE}")
    X_emb = np.load(EMBEDDINGS_CACHE)
    df_emb_ids = pd.read_parquet(EMBEDDINGS_IDS_CACHE)
    print(f"  Shape: {X_emb.shape}")
else:
    print("Loading model (first run downloads ~80MB)...")
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    print(f"Model loaded. Embedding dim: {model.get_sentence_embedding_dimension()}")
    
    df_split = pd.read_parquet(SPLIT_CACHE)
    texts = df_split["text"].tolist()
    
    print(f"\nEncoding {len(texts):,} sentences (this takes ~5-15 min on CPU)...")
    X_emb = model.encode(
        texts,
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,  # makes cosine == dot product later
    )
    print(f"  Done. Shape: {X_emb.shape}")
    
    np.save(EMBEDDINGS_CACHE, X_emb)
    df_split[["sentence_id"]].to_parquet(EMBEDDINGS_IDS_CACHE, index=False)
    print(f"\n✅ Saved → {EMBEDDINGS_CACHE}")

# Quick sanity check: embeddings should be unit-normalized
print(f"\nSanity check (norms should ≈ 1.0): {np.linalg.norm(X_emb, axis=1).mean():.4f}")
print(f"Embedding stats: min={X_emb.min():.3f}, max={X_emb.max():.3f}, mean={X_emb.mean():.4f}")

Loading cached embeddings from C:\Users\loren\Desktop\MTH9796\HW2\cache\embeddings.npy
  Shape: (2348, 384)

Sanity check (norms should ≈ 1.0): 1.0000
Embedding stats: min=-0.284, max=0.238, mean=-0.0003


In [23]:
# Cell 16: Unified feature matrices for train/val/test

import numpy as np

df_split = pd.read_parquet(SPLIT_CACHE)
df_features_regex = pd.read_parquet(FEATURES_CACHE)
X_emb = np.load(EMBEDDINGS_CACHE)

# Sanity check: same order
assert (df_split["sentence_id"].values == df_features_regex["sentence_id"].values).all()

# Build label vector (binary: 1 = BOILERPLATE, 0 = SUBSTANTIVE)
# We classify boilerplate as the positive class for clarity in metrics.
# But the rubric cares about SUBSTANTIVE recall, which equals BOILERPLATE specificity.
# To keep the math simple we'll model SUBSTANTIVE as positive and BOILERPLATE as negative,
# matching the rubric's framing (substantive recall ≥ 0.96).
df_split["y"] = (df_split["gold_label"] == "SUBSTANTIVE").astype(int)

# Regex feature matrix (drop sentence_id)
X_regex = df_features_regex.drop(columns=["sentence_id"]).values.astype(np.float32)

# Combined matrix: embedding + regex
X_combined = np.hstack([X_emb, X_regex])

print(f"Embedding matrix:  X_emb     shape = {X_emb.shape}")
print(f"Regex matrix:      X_regex   shape = {X_regex.shape}")
print(f"Combined matrix:   X_combined shape = {X_combined.shape}")

# Slice into train/val/test
def get_split(name):
    mask = df_split["split"] == name
    return {
        "y": df_split.loc[mask, "y"].values,
        "text": df_split.loc[mask, "text"].values,
        "sentence_id": df_split.loc[mask, "sentence_id"].values,
        "X_emb": X_emb[mask.values],
        "X_regex": X_regex[mask.values],
        "X_combined": X_combined[mask.values],
    }

splits = {name: get_split(name) for name in ["train", "val", "test"]}
print(f"\n=== Split shapes ===")
for name, s in splits.items():
    pos = (s["y"] == 1).sum()
    neg = (s["y"] == 0).sum()
    print(f"  {name:5s}: {len(s['y']):4d} sentences  ({pos} SUBS, {neg} BOIL)")

print("\n✅ Feature matrices ready. Splits stored in `splits` dict.")

Embedding matrix:  X_emb     shape = (2348, 384)
Regex matrix:      X_regex   shape = (2348, 29)
Combined matrix:   X_combined shape = (2348, 413)

=== Split shapes ===
  train: 1408 sentences  (1245 SUBS, 163 BOIL)
  val  :  470 sentences  (415 SUBS, 55 BOIL)
  test :  470 sentences  (416 SUBS, 54 BOIL)

✅ Feature matrices ready. Splits stored in `splits` dict.


In [24]:
# Cell 17: Classifier registry + Rules-only baseline

import time
import pickle
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix
)

CLASSIFIERS_CACHE = MODELS_DIR / "classifier_results.pkl"

# Classifier registry — each entry stores predictions, probabilities, timings, and params
# Keys: classifier_name; Values: dict with "val_proba", "test_proba", "train_time_sec", "throughput_sent_per_sec", "model_obj" (optional)
classifier_results = {}

if CLASSIFIERS_CACHE.exists():
    with open(CLASSIFIERS_CACHE, "rb") as f:
        classifier_results = pickle.load(f)
    print(f"Loaded existing results: {list(classifier_results.keys())}")


def evaluate_classifier(name, val_proba, test_proba, train_time, throughput, save=True):
    """Print val/test metrics at default 0.5 threshold + cache results.
    
    NOTE: We're modeling SUBSTANTIVE as the positive class. So val_proba/test_proba
    are P(SUBSTANTIVE). At threshold 0.5, predict SUBS if P >= 0.5.
    Substantive recall (rubric's hard floor) = recall of the positive class.
    """
    y_val = splits["val"]["y"]
    y_test = splits["test"]["y"]
    
    # Default threshold for this preview (we'll tune properly in Cell 24)
    val_pred = (val_proba >= 0.5).astype(int)
    test_pred = (test_proba >= 0.5).astype(int)
    
    metrics = {
        "val_acc": accuracy_score(y_val, val_pred),
        "val_macro_f1": f1_score(y_val, val_pred, average="macro"),
        "val_subs_recall": recall_score(y_val, val_pred, pos_label=1),
        "val_boil_f1": f1_score(y_val, val_pred, pos_label=0),
        "test_acc": accuracy_score(y_test, test_pred),
        "test_macro_f1": f1_score(y_test, test_pred, average="macro"),
        "test_subs_recall": recall_score(y_test, test_pred, pos_label=1),
        "test_boil_f1": f1_score(y_test, test_pred, pos_label=0),
        "train_time_sec": train_time,
        "throughput_sent_per_sec": throughput,
    }
    
    classifier_results[name] = {
        "val_proba": val_proba,
        "test_proba": test_proba,
        **metrics,
    }
    
    if save:
        with open(CLASSIFIERS_CACHE, "wb") as f:
            pickle.dump(classifier_results, f)
    
    print(f"=== {name} ===")
    print(f"  Train time: {train_time:.2f}s    Throughput: {throughput:.0f} sent/s")
    print(f"  VAL : acc={metrics['val_acc']:.3f}  macro-F1={metrics['val_macro_f1']:.3f}  "
          f"subs-recall={metrics['val_subs_recall']:.3f}  boil-F1={metrics['val_boil_f1']:.3f}")
    print(f"  TEST: acc={metrics['test_acc']:.3f}  macro-F1={metrics['test_macro_f1']:.3f}  "
          f"subs-recall={metrics['test_subs_recall']:.3f}  boil-F1={metrics['test_boil_f1']:.3f}")


# ============================================================
# Classifier #1: Rules-only baseline
# ============================================================
# A purely rule-based classifier that uses the regex flags directly.
# Decision logic: if any strong-BOIL flag fires AND no strong-SUBS flag fires, predict BOIL.
# Otherwise predict SUBS (lenient default — favors substantive recall).

print("Training rules-only baseline...")
t0 = time.time()

BOIL_FLAGS = [
    "f_operator_intro", "f_speaker_intro_with_firm", "f_operator_instructions",
    "f_safe_harbor", "f_replay_recording", "f_greeting", "f_call_handoff",
    "f_appreciate_question", "f_congrats", "f_section_transition", "f_closing_remarks",
]
SUBS_FLAGS = [
    "f_dollar_amount", "f_percent", "f_basis_points", "f_year_over_year",
    "f_growth_decline", "f_margin_term",
]

# Get column indices
feat_cols = list(df_features_regex.drop(columns=["sentence_id"]).columns)
boil_idx = [feat_cols.index(f) for f in BOIL_FLAGS]
subs_idx = [feat_cols.index(f) for f in SUBS_FLAGS]

def rule_classifier(X_regex):
    """Returns P(SUBSTANTIVE) ∈ {0.1, 0.5, 0.9} for each row."""
    has_boil = X_regex[:, boil_idx].sum(axis=1) > 0
    has_subs = X_regex[:, subs_idx].sum(axis=1) > 0
    # If only boil signals fire → strong BOIL prediction (low P_subs)
    # If subs signals fire → strong SUBS prediction (high P_subs)
    # Otherwise → neutral default (slightly subs-leaning to favor recall)
    proba = np.full(len(X_regex), 0.7, dtype=float)
    proba[has_boil & ~has_subs] = 0.1
    proba[has_subs] = 0.95
    return proba

# Apply
val_proba = rule_classifier(splits["val"]["X_regex"])
test_proba = rule_classifier(splits["test"]["X_regex"])

train_time = time.time() - t0
throughput = (len(splits["val"]["y"]) + len(splits["test"]["y"])) / max(0.001, train_time)

evaluate_classifier("rules_only", val_proba, test_proba, train_time, throughput)

Loaded existing results: ['rules_only', 'logreg_emb', 'histgbm_combined', 'svm_rbf_emb', 'tfidf_ngram', 'mlp_emb', 'ensemble_mean', 'ensemble_rank', 'floret_real', 'finbert_finetuned']
Training rules-only baseline...
=== rules_only ===
  Train time: 0.00s    Throughput: 432640 sent/s
  VAL : acc=0.936  macro-F1=0.808  subs-recall=0.993  boil-F1=0.651
  TEST: acc=0.930  macro-F1=0.782  subs-recall=0.990  boil-F1=0.602


In [25]:
# Cell 18: Classifier #2 — Logistic Regression on frozen embeddings

from sklearn.linear_model import LogisticRegression

print("Training logistic regression on embeddings...")
t0 = time.time()

X_train = splits["train"]["X_emb"]
y_train = splits["train"]["y"]

clf = LogisticRegression(
    C=1.0,
    max_iter=2000,
    class_weight="balanced",  # important — minority class is BOIL
    random_state=RANDOM_SEED,
)
clf.fit(X_train, y_train)

train_time = time.time() - t0

# Predictions (P(SUBSTANTIVE))
t_inf = time.time()
val_proba = clf.predict_proba(splits["val"]["X_emb"])[:, 1]
test_proba = clf.predict_proba(splits["test"]["X_emb"])[:, 1]
inf_time = time.time() - t_inf
throughput = (len(splits["val"]["y"]) + len(splits["test"]["y"])) / max(0.001, inf_time)

evaluate_classifier("logreg_emb", val_proba, test_proba, train_time, throughput)

Training logistic regression on embeddings...
=== logreg_emb ===
  Train time: 0.01s    Throughput: 548275 sent/s
  VAL : acc=0.891  macro-F1=0.794  subs-recall=0.894  boil-F1=0.653
  TEST: acc=0.909  macro-F1=0.809  subs-recall=0.921  boil-F1=0.672


In [26]:
# Cell 19: Classifier #3 — Gradient-boosted trees on embeddings + regex flags

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight

print("Training HistGBM on combined features...")
t0 = time.time()

X_train = splits["train"]["X_combined"]
y_train = splits["train"]["y"]
sample_weight = compute_sample_weight("balanced", y_train)

clf = HistGradientBoostingClassifier(
    max_iter=300,
    max_leaf_nodes=31,
    learning_rate=0.1,
    random_state=RANDOM_SEED,
)
clf.fit(X_train, y_train, sample_weight=sample_weight)

train_time = time.time() - t0

t_inf = time.time()
val_proba = clf.predict_proba(splits["val"]["X_combined"])[:, 1]
test_proba = clf.predict_proba(splits["test"]["X_combined"])[:, 1]
inf_time = time.time() - t_inf
throughput = (len(splits["val"]["y"]) + len(splits["test"]["y"])) / max(0.001, inf_time)

evaluate_classifier("histgbm_combined", val_proba, test_proba, train_time, throughput)

Training HistGBM on combined features...
=== histgbm_combined ===
  Train time: 6.85s    Throughput: 23754 sent/s
  VAL : acc=0.955  macro-F1=0.879  subs-recall=0.990  boil-F1=0.784
  TEST: acc=0.940  macro-F1=0.824  subs-recall=0.990  boil-F1=0.682


In [27]:
# Cell 20: Classifier #4 — SVM with RBF kernel on embeddings

from sklearn.svm import SVC

print("Training SVM-RBF on embeddings...")
t0 = time.time()

X_train = splits["train"]["X_emb"]
y_train = splits["train"]["y"]

clf = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    probability=True,  # needed for proba output
    class_weight="balanced",
    random_state=RANDOM_SEED,
)
clf.fit(X_train, y_train)

train_time = time.time() - t0

t_inf = time.time()
val_proba = clf.predict_proba(splits["val"]["X_emb"])[:, 1]
test_proba = clf.predict_proba(splits["test"]["X_emb"])[:, 1]
inf_time = time.time() - t_inf
throughput = (len(splits["val"]["y"]) + len(splits["test"]["y"])) / max(0.001, inf_time)

evaluate_classifier("svm_rbf_emb", val_proba, test_proba, train_time, throughput)

Training SVM-RBF on embeddings...
=== svm_rbf_emb ===
  Train time: 0.45s    Throughput: 11763 sent/s
  VAL : acc=0.947  macro-F1=0.870  subs-recall=0.971  boil-F1=0.771
  TEST: acc=0.949  macro-F1=0.863  subs-recall=0.983  boil-F1=0.755


In [28]:
# Cell 21: Classifier #5 — FastText (or scikit-learn fallback) on raw text n-grams
# 
# True FastText is a separate library. For simplicity (and because it often has install issues),
# we use scikit-learn's TF-IDF + LogisticRegression on character + word n-grams as the
# functional equivalent. This is a "bag-of-ngrams" model — no embeddings, no semantic understanding.

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

print("Training TF-IDF + LogReg (FastText-equivalent) on raw text...")
t0 = time.time()

train_texts = splits["train"]["text"]
val_texts = splits["val"]["text"]
test_texts = splits["test"]["text"]
y_train = splits["train"]["y"]

# TF-IDF over both word (1-2) and character (3-5) n-grams to capture both vocab and morphology
clf = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
    )),
    ("lr", LogisticRegression(
        C=1.0,
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_SEED,
    )),
])

clf.fit(train_texts, y_train)
train_time = time.time() - t0

t_inf = time.time()
val_proba = clf.predict_proba(val_texts)[:, 1]
test_proba = clf.predict_proba(test_texts)[:, 1]
inf_time = time.time() - t_inf
throughput = (len(val_texts) + len(test_texts)) / max(0.001, inf_time)

evaluate_classifier("tfidf_ngram", val_proba, test_proba, train_time, throughput)

Training TF-IDF + LogReg (FastText-equivalent) on raw text...
=== tfidf_ngram ===
  Train time: 0.07s    Throughput: 33528 sent/s
  VAL : acc=0.943  macro-F1=0.858  subs-recall=0.971  boil-F1=0.748
  TEST: acc=0.940  macro-F1=0.835  subs-recall=0.983  boil-F1=0.702


In [29]:
# Cell 21b: Classifier #5b — Real floret (FastText-family) on raw text
#
# floret is spaCy's actively-maintained reimplementation of FastText. It trains
# subword character n-gram embeddings (3-6 chars) plus a softmax classifier in
# one shot. Different family from TF-IDF + LogReg (sparse word n-grams + linear).
#
# This cell satisfies the handout's "FastText / floret" leaderboard slot.
# (Real fasttext didn't compile on Python 3.14 due to a C++17 declaration issue
# upstream; floret installs cleanly from a wheel.)

import floret
import tempfile
import os

print("Training floret on raw text...")
t0 = time.time()

train_texts = splits["train"]["text"]
val_texts = splits["val"]["text"]
test_texts = splits["test"]["text"]
y_train = splits["train"]["y"]

label_str = lambda y: "__label__SUBS" if y == 1 else "__label__BOIL"

# floret reads a labeled file in fasttext format: __label__<class> <text>
with tempfile.NamedTemporaryFile("w", suffix=".txt", delete=False, encoding="utf-8") as f:
    train_file = f.name
    for text, y in zip(train_texts, y_train):
        clean = text.replace("\n", " ").replace("\r", " ").strip()
        f.write(f"{label_str(y)} {clean}\n")

# wordNgrams=2 + subword n-grams 3-6 is the standard config for short text.
# mode="fasttext" gives the original fasttext-style hashing (vs floret's bucket mode).
ft_model = floret.train_supervised(
    input=train_file,
    epoch=25,
    lr=0.5,
    wordNgrams=2,
    minn=3,
    maxn=6,
    dim=100,
    loss="softmax",
    mode="fasttext",
    verbose=0,
)

os.unlink(train_file)
train_time = time.time() - t0


def floret_predict_proba(model, texts):
    """Return P(SUBSTANTIVE) for each text. Bypasses floret's buggy np.array(copy=False) call."""
    probs = np.zeros(len(texts))
    for i, t in enumerate(texts):
        clean = t.replace("\n", " ").replace("\r", " ").strip()
        # Call the underlying C++ predict directly, skip floret's Python wrapper
        # Signature: (text, top_k, threshold, on_unicode_error) -> list[(prob, label)]
        results = model.f.predict(clean, 2, 0.0, "strict")
        for prob, lbl in results:
            if lbl == "__label__SUBS":
                probs[i] = prob
                break
    return probs


t_inf = time.time()
val_proba = floret_predict_proba(ft_model, val_texts)
test_proba = floret_predict_proba(ft_model, test_texts)
inf_time = time.time() - t_inf
throughput = (len(val_texts) + len(test_texts)) / max(0.001, inf_time)

evaluate_classifier("floret_real", val_proba, test_proba, train_time, throughput)

Training floret on raw text...
=== floret_real ===
  Train time: 1.30s    Throughput: 7305 sent/s
  VAL : acc=0.957  macro-F1=0.886  subs-recall=0.990  boil-F1=0.796
  TEST: acc=0.947  macro-F1=0.845  subs-recall=0.993  boil-F1=0.719


In [30]:
'''
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "pyarrow", "datasets", "-q"])
print("Done. Restart the kernel now.")
'''

'\nimport subprocess, sys\nsubprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "pyarrow", "datasets", "-q"])\nprint("Done. Restart the kernel now.")\n'

In [31]:
# Cell 22: Classifier #6 — MLP on embeddings (replaces SetFit)
#
# NOTE: We originally planned to include SetFit (contrastive fine-tuning) here.
# However, SetFit and its dependency stack (transformers, datasets) had
# unresolved compatibility issues on Python 3.14 at the time of this run.
# We substitute a multi-layer perceptron classifier on the same frozen
# sentence embeddings — this captures the same "non-linear neural classifier
# on dense representations" family without the dependency conflicts.

from sklearn.neural_network import MLPClassifier

print("Training MLP on embeddings...")
t0 = time.time()

X_train = splits["train"]["X_emb"]
y_train = splits["train"]["y"]

# Two-layer MLP with class weighting via sample_weight workaround
# (MLPClassifier doesn't support class_weight directly)
from sklearn.utils.class_weight import compute_sample_weight

clf = MLPClassifier(
    hidden_layer_sizes=(128, 32),
    activation="relu",
    solver="adam",
    alpha=1e-4,
    batch_size=64,
    learning_rate_init=1e-3,
    max_iter=200,
    early_stopping=True,
    validation_fraction=0.15,
    random_state=RANDOM_SEED,
)
clf.fit(X_train, y_train)

train_time = time.time() - t0

t_inf = time.time()
val_proba = clf.predict_proba(splits["val"]["X_emb"])[:, 1]
test_proba = clf.predict_proba(splits["test"]["X_emb"])[:, 1]
inf_time = time.time() - t_inf
throughput = (len(splits["val"]["y"]) + len(splits["test"]["y"])) / max(0.001, inf_time)

evaluate_classifier("mlp_emb", val_proba, test_proba, train_time, throughput)

Training MLP on embeddings...
=== mlp_emb ===
  Train time: 3.99s    Throughput: 172825 sent/s
  VAL : acc=0.945  macro-F1=0.866  subs-recall=0.969  boil-F1=0.764
  TEST: acc=0.936  macro-F1=0.843  subs-recall=0.964  boil-F1=0.722


In [32]:
# Cell 22b: Classifier #6b — Fine-tuned FinBERT
#
# We fine-tune ProsusAI/finbert (a financial-domain BERT) with a fresh
# 2-class head on our gold labels. This is the handout's "Fine-tuned
# transformer (FinBERT)" leaderboard family.
#
# Cached: if cache/finbert_probas.npz exists, we load val_proba and test_proba
# from disk and skip the ~22-minute training. Delete the cache file to force
# a fresh fit.

import os
# Force PyTorch's threading to a sane configuration BEFORE importing torch.
os.environ["OMP_NUM_THREADS"] = str(max(1, os.cpu_count() // 2))
os.environ["MKL_NUM_THREADS"] = str(max(1, os.cpu_count() // 2))

import torch
torch.set_num_threads(max(1, os.cpu_count() // 2))
# torch.set_num_interop_threads(2)  # can only be set once per process; skip

from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

print(f"  CPU cores available: {os.cpu_count()}")
print(f"  torch.get_num_threads(): {torch.get_num_threads()}")
print(f"  torch.get_num_interop_threads(): {torch.get_num_interop_threads()}")

FINBERT_PROBA_CACHE = CACHE_DIR / "finbert_probas.npz"

if FINBERT_PROBA_CACHE.exists():
    print(f"Loading cached FinBERT predictions from {FINBERT_PROBA_CACHE}")
    cached = np.load(FINBERT_PROBA_CACHE)
    val_proba = cached["val_proba"]
    test_proba = cached["test_proba"]
    train_time = float(cached["train_time"])
    throughput = float(cached["throughput"])
    evaluate_classifier("finbert_finetuned", val_proba, test_proba, train_time, throughput)
    print("✅ Cached FinBERT loaded; skipping training.")

else:
    # --- Train from scratch ---
    print("Fine-tuning FinBERT (ProsusAI/finbert) on gold labels...")
    t0 = time.time()

    MODEL_NAME = "ProsusAI/finbert"
    MAX_LEN = 96
    BATCH_SIZE = 32
    N_EPOCHS = 2
    LR = 2e-5
    WARMUP_FRAC = 0.1

    torch.manual_seed(RANDOM_SEED)
    torch.use_deterministic_algorithms(False)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"  Device: {device}")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


    class SentenceDataset(Dataset):
        def __init__(self, texts, labels=None):
            self.texts = list(texts)
            self.labels = list(labels) if labels is not None else None
            self.enc = tokenizer(
                self.texts, truncation=True, padding="max_length",
                max_length=MAX_LEN, return_tensors="pt",
            )

        def __len__(self):
            return len(self.texts)

        def __getitem__(self, idx):
            item = {k: v[idx] for k, v in self.enc.items()}
            if self.labels is not None:
                item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
            return item


    train_ds = SentenceDataset(splits["train"]["text"], splits["train"]["y"])
    val_ds   = SentenceDataset(splits["val"]["text"],   splits["val"]["y"])
    test_ds  = SentenceDataset(splits["test"]["text"],  splits["test"]["y"])

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=False)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2, ignore_mismatched_sizes=True,
    )
    model.to(device)

    y_train_arr = np.array(splits["train"]["y"])
    n_pos = (y_train_arr == 1).sum()
    n_neg = (y_train_arr == 0).sum()
    n_total = len(y_train_arr)
    class_weights = torch.tensor(
        [n_total / (2 * n_neg), n_total / (2 * n_pos)],
        dtype=torch.float32,
    ).to(device)
    loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total_steps = len(train_loader) * N_EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(WARMUP_FRAC * total_steps),
        num_training_steps=total_steps,
    )

    model.train()
    for epoch in range(N_EPOCHS):
        epoch_loss = 0.0
        for step, batch in enumerate(train_loader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            optimizer.zero_grad()
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = loss_fn(outputs.logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            epoch_loss += loss.item()
        print(f"  Epoch {epoch+1}/{N_EPOCHS}: avg loss = {epoch_loss / len(train_loader):.4f}")

    train_time = time.time() - t0


    def finbert_predict_proba(loader):
        model.eval()
        probs = []
        with torch.no_grad():
            for batch in loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
                p = torch.softmax(logits, dim=-1)[:, 1]
                probs.extend(p.cpu().numpy().tolist())
        return np.array(probs)


    t_inf = time.time()
    val_proba = finbert_predict_proba(val_loader)
    test_proba = finbert_predict_proba(test_loader)
    inf_time = time.time() - t_inf
    throughput = (len(val_ds) + len(test_ds)) / max(0.001, inf_time)

    FINBERT_DIR = MODELS_DIR / "finbert_finetuned"
    FINBERT_DIR.mkdir(exist_ok=True)
    model.save_pretrained(FINBERT_DIR)
    tokenizer.save_pretrained(FINBERT_DIR)
    print(f"  Saved fine-tuned FinBERT → {FINBERT_DIR}")

    # Cache predictions for future runs
    np.savez(
        FINBERT_PROBA_CACHE,
        val_proba=val_proba,
        test_proba=test_proba,
        train_time=train_time,
        throughput=throughput,
    )
    print(f"  Saved FinBERT predictions → {FINBERT_PROBA_CACHE}")

    evaluate_classifier("finbert_finetuned", val_proba, test_proba, train_time, throughput)

  CPU cores available: 12
  torch.get_num_threads(): 6
  torch.get_num_interop_threads(): 10
Loading cached FinBERT predictions from C:\Users\loren\Desktop\MTH9796\HW2\cache\finbert_probas.npz
=== finbert_finetuned ===
  Train time: 1308.21s    Throughput: 7 sent/s
  VAL : acc=0.911  macro-F1=0.828  subs-recall=0.908  boil-F1=0.708
  TEST: acc=0.930  macro-F1=0.852  subs-recall=0.935  boil-F1=0.744
✅ Cached FinBERT loaded; skipping training.


In [33]:
import numpy as np
FINBERT_PROBA_CACHE = CACHE_DIR / "finbert_probas.npz"
np.savez(
    FINBERT_PROBA_CACHE,
    val_proba=val_proba,
    test_proba=test_proba,
    train_time=train_time,
    throughput=throughput,
)
print(f"✅ Saved FinBERT predictions → {FINBERT_PROBA_CACHE}")
print(f"   File size: {FINBERT_PROBA_CACHE.stat().st_size / 1024:.1f} KB")

✅ Saved FinBERT predictions → C:\Users\loren\Desktop\MTH9796\HW2\cache\finbert_probas.npz
   File size: 8.4 KB


In [34]:
# Cell 23: Classifier #7 & #8 — Ensembles of top-5 base models
#
# Two ensembles, both built from the top-5 non-rules models:
#   - mean_prob_ensemble: average the P(SUBS) probabilities
#   - rank_avg_ensemble:  average the rank-orders, then map back to a probability
# Rank-averaged is more robust when models have very different probability calibrations.

from scipy.stats import rankdata

# Pick the top-5 base classifiers we want to ensemble (excluding rules-only and ensembles themselves)
ENSEMBLE_MEMBERS = [
    "svm_rbf_emb",
    "mlp_emb",
    "tfidf_ngram",
    "histgbm_combined",
    "logreg_emb",
]

print(f"Ensembling: {ENSEMBLE_MEMBERS}\n")

# Stack member probabilities for val and test
val_probas = np.column_stack([classifier_results[m]["val_proba"] for m in ENSEMBLE_MEMBERS])
test_probas = np.column_stack([classifier_results[m]["test_proba"] for m in ENSEMBLE_MEMBERS])

# --- Mean-probability ensemble ---
print("Building mean-probability ensemble...")
t0 = time.time()
val_mean = val_probas.mean(axis=1)
test_mean = test_probas.mean(axis=1)
ensemble_time = time.time() - t0
throughput = (len(splits["val"]["y"]) + len(splits["test"]["y"])) / max(0.001, ensemble_time)
evaluate_classifier("ensemble_mean", val_mean, test_mean, ensemble_time, throughput)

# --- Rank-averaged ensemble ---
print("\nBuilding rank-averaged ensemble...")
t0 = time.time()

# Rank each member's probabilities (1 = lowest, N = highest)
# then average ranks and rescale to [0, 1] like a probability
val_ranks = np.column_stack([rankdata(val_probas[:, i]) for i in range(val_probas.shape[1])])
test_ranks = np.column_stack([rankdata(test_probas[:, i]) for i in range(test_probas.shape[1])])

val_rank_avg = val_ranks.mean(axis=1) / len(splits["val"]["y"])
test_rank_avg = test_ranks.mean(axis=1) / len(splits["test"]["y"])

ensemble_time = time.time() - t0
throughput = (len(splits["val"]["y"]) + len(splits["test"]["y"])) / max(0.001, ensemble_time)
evaluate_classifier("ensemble_rank", val_rank_avg, test_rank_avg, ensemble_time, throughput)

Ensembling: ['svm_rbf_emb', 'mlp_emb', 'tfidf_ngram', 'histgbm_combined', 'logreg_emb']

Building mean-probability ensemble...
=== ensemble_mean ===
  Train time: 0.00s    Throughput: 798268 sent/s
  VAL : acc=0.951  macro-F1=0.879  subs-recall=0.976  boil-F1=0.785
  TEST: acc=0.945  macro-F1=0.852  subs-recall=0.981  boil-F1=0.735

Building rank-averaged ensemble...
=== ensemble_rank ===
  Train time: 0.00s    Throughput: 194056 sent/s
  VAL : acc=0.649  macro-F1=0.576  subs-recall=0.602  boil-F1=0.400
  TEST: acc=0.630  macro-F1=0.559  subs-recall=0.582  boil-F1=0.383


### Note on the rank-averaged ensemble

The rank-averaged ensemble's macro-F1 looks dramatically worse than the mean-probability 
ensemble at the default 0.5 threshold (0.56 vs 0.85), but this is **a calibration artifact, 
not a true performance gap**.

Rank-averaging produces probabilities that are uniformly distributed over [0, 1] by 
construction: the rank of the i-th sample divided by N. This means a 0.5 threshold 
classifies roughly half the data as boilerplate, which is wildly miscalibrated for our 
~88/12 class balance — most actual SUBSTANTIVE sentences end up below 0.5 and get 
misclassified.

The fix is **threshold tuning**, which is the next step (Cell 24). Once we tune the 
threshold to enforce substantive-recall ≥ 0.96 via 5-fold OOF, the rank-averaged 
ensemble will be evaluated at its proper operating point and we can compare it 
fairly to the other models.

This is also a good example of why the rubric requires threshold tuning rather than 
default-0.5 evaluation: probabilistic outputs from different model families have very 
different calibration scales, and the "right" threshold for one is rarely 0.5.

In [35]:
# Cell 24: Recall-constrained threshold tuning via 5-fold OOF
#
# We refit each classifier with 5-fold cross-validation on the train+val pool
# to get out-of-fold (OOF) probabilities, then sweep thresholds to find the
# one that maximizes macro-F1 subject to substantive recall ≥ 0.96.
#
# OOF probabilities are cached to cache/oof_probas.npz. On subsequent runs,
# every classifier loads from disk in milliseconds. Delete the file to force
# recomputation.

from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_sample_weight

RECALL_FLOOR = 0.96

# --- Cache helpers ---
OOF_CACHE = CACHE_DIR / "oof_probas.npz"


def load_oof_cache():
    if OOF_CACHE.exists():
        d = np.load(OOF_CACHE)
        return {k: d[k] for k in d.files}
    return {}


def save_oof_cache(oof_dict):
    np.savez(OOF_CACHE, **oof_dict)


# Combine train + val for cross-validation
trainval_mask = (df_split["split"] == "train") | (df_split["split"] == "val")
X_emb_tv = X_emb[trainval_mask.values]
X_combined_tv = X_combined[trainval_mask.values]
texts_tv = df_split.loc[trainval_mask, "text"].values
y_tv = df_split.loc[trainval_mask, "y"].values

print(f"Cross-validation pool: {len(y_tv):,} sentences (train + val)")
print(f"  SUBS: {(y_tv == 1).sum()}  BOIL: {(y_tv == 0).sum()}")


# Define how to refit each model in a fold
def fit_predict_emb_logreg(X_train, y_train, X_test):
    clf = LogisticRegression(C=1.0, max_iter=2000, class_weight="balanced", random_state=RANDOM_SEED)
    clf.fit(X_train, y_train)
    return clf.predict_proba(X_test)[:, 1]

def fit_predict_svm(X_train, y_train, X_test):
    clf = SVC(kernel="rbf", C=1.0, gamma="scale", probability=True, class_weight="balanced", random_state=RANDOM_SEED)
    clf.fit(X_train, y_train)
    return clf.predict_proba(X_test)[:, 1]

def fit_predict_histgbm(X_train, y_train, X_test):
    sw = compute_sample_weight("balanced", y_train)
    clf = HistGradientBoostingClassifier(max_iter=300, max_leaf_nodes=31, learning_rate=0.1, random_state=RANDOM_SEED)
    clf.fit(X_train, y_train, sample_weight=sw)
    return clf.predict_proba(X_test)[:, 1]

def fit_predict_mlp(X_train, y_train, X_test):
    clf = MLPClassifier(hidden_layer_sizes=(128, 32), activation="relu", solver="adam",
                        alpha=1e-4, batch_size=64, learning_rate_init=1e-3, max_iter=200,
                        early_stopping=True, validation_fraction=0.15, random_state=RANDOM_SEED)
    clf.fit(X_train, y_train)
    return clf.predict_proba(X_test)[:, 1]

def fit_predict_tfidf(X_train_text, y_train, X_test_text):
    clf = Pipeline([
        ("tfidf", TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True)),
        ("lr", LogisticRegression(C=1.0, max_iter=2000, class_weight="balanced", random_state=RANDOM_SEED)),
    ])
    clf.fit(X_train_text, y_train)
    return clf.predict_proba(X_test_text)[:, 1]

def fit_predict_floret(X_train_text, y_train, X_test_text):
    import floret, tempfile, os
    with tempfile.NamedTemporaryFile("w", suffix=".txt", delete=False, encoding="utf-8") as f:
        train_file = f.name
        for text, y in zip(X_train_text, y_train):
            clean = text.replace("\n", " ").replace("\r", " ").strip()
            lbl = "__label__SUBS" if y == 1 else "__label__BOIL"
            f.write(f"{lbl} {clean}\n")
    m = floret.train_supervised(
        input=train_file, epoch=25, lr=0.5, wordNgrams=2,
        minn=3, maxn=6, dim=100, loss="softmax", mode="fasttext", verbose=0,
    )
    os.unlink(train_file)
    probs = np.zeros(len(X_test_text))
    for i, t in enumerate(X_test_text):
        clean = t.replace("\n", " ").replace("\r", " ").strip()
        results = m.f.predict(clean, 2, 0.0, "strict")
        for prob, lbl in results:
            if lbl == "__label__SUBS":
                probs[i] = prob
                break
    return probs


def fit_predict_finbert(X_train_text, y_train, X_test_text):
    """5-fold OOF helper for FinBERT."""
    import os
    import torch
    from torch.utils.data import Dataset as _Dataset, DataLoader as _DataLoader
    from transformers import (
        AutoTokenizer,
        AutoModelForSequenceClassification,
        get_linear_schedule_with_warmup,
    )

    MODEL_NAME = "ProsusAI/finbert"
    MAX_LEN = 96
    BATCH_SIZE = 32
    N_EPOCHS = 2
    LR = 2e-5
    WARMUP_FRAC = 0.1

    torch.manual_seed(RANDOM_SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    tok = AutoTokenizer.from_pretrained(MODEL_NAME)

    class _DS(_Dataset):
        def __init__(self, texts, labels=None):
            self.enc = tok(
                list(texts), truncation=True, padding="max_length",
                max_length=MAX_LEN, return_tensors="pt",
            )
            self.labels = list(labels) if labels is not None else None

        def __len__(self):
            return len(self.enc["input_ids"])

        def __getitem__(self, idx):
            item = {k: v[idx] for k, v in self.enc.items()}
            if self.labels is not None:
                item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
            return item

    train_loader = _DataLoader(
        _DS(X_train_text, y_train),
        batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=False,
    )
    test_loader = _DataLoader(
        _DS(X_test_text),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False,
    )

    m = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2, ignore_mismatched_sizes=True,
    ).to(device)

    n_pos = sum(1 for y in y_train if y == 1)
    n_neg = sum(1 for y in y_train if y == 0)
    n_total = len(y_train)
    class_weights = torch.tensor(
        [n_total / (2 * n_neg), n_total / (2 * n_pos)],
        dtype=torch.float32,
    ).to(device)
    loss_fn_local = torch.nn.CrossEntropyLoss(weight=class_weights)

    optimizer = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=0.01)
    total_steps = len(train_loader) * N_EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(WARMUP_FRAC * total_steps),
        num_training_steps=total_steps,
    )

    m.train()
    for epoch in range(N_EPOCHS):
        for batch in train_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            optimizer.zero_grad()
            logits = m(input_ids=input_ids, attention_mask=attention_mask).logits
            loss = loss_fn_local(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()

    m.eval()
    probs = []
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            logits = m(input_ids=input_ids, attention_mask=attention_mask).logits
            p = torch.softmax(logits, dim=-1)[:, 1]
            probs.extend(p.cpu().numpy().tolist())

    del m, optimizer, scheduler, train_loader, test_loader
    if device.type == "cuda":
        torch.cuda.empty_cache()

    return np.array(probs)


# --- Run 5-fold CV per classifier (cached) ---
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

oof_proba = load_oof_cache()
print(f"\nLoaded {len(oof_proba)} cached OOF probabilities: {list(oof_proba.keys())}")

CLASSIFIER_FITS = {
    "logreg_emb":        (X_emb_tv,      fit_predict_emb_logreg),
    "svm_rbf_emb":       (X_emb_tv,      fit_predict_svm),
    "histgbm_combined":  (X_combined_tv, fit_predict_histgbm),
    "mlp_emb":           (X_emb_tv,      fit_predict_mlp),
    "tfidf_ngram":       (texts_tv,      fit_predict_tfidf),
    "floret_real":       (texts_tv,      fit_predict_floret),
    "finbert_finetuned": (texts_tv,      fit_predict_finbert),
}

for name, (X_tv, fit_fn) in CLASSIFIER_FITS.items():
    if name in oof_proba and len(oof_proba[name]) == len(y_tv):
        print(f"  CV: {name} (cached, skipping)")
        continue

    print(f"  CV: {name} (fitting 5 folds — this may take a while)")
    proba = np.zeros(len(y_tv))
    for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(np.zeros(len(y_tv)), y_tv)):
        print(f"    Fold {fold_idx + 1}/5...")
        X_tr = X_tv[tr_idx]
        y_tr = y_tv[tr_idx]
        X_te = X_tv[te_idx]
        proba[te_idx] = fit_fn(X_tr, y_tr, X_te)
    oof_proba[name] = proba
    # Persist immediately after each classifier finishes — crash-safe
    save_oof_cache(oof_proba)
    print(f"    ✅ {name} cached.")

print(f"\n✅ All OOF probabilities cached in {OOF_CACHE}")

# --- Add ensembles (built from OOF members; arrays only, milliseconds) ---
oof_members = ["svm_rbf_emb", "mlp_emb", "tfidf_ngram", "histgbm_combined", "logreg_emb"]
oof_stack = np.column_stack([oof_proba[m] for m in oof_members])
oof_proba["ensemble_mean"] = oof_stack.mean(axis=1)
from scipy.stats import rankdata
oof_ranks = np.column_stack([rankdata(oof_stack[:, i]) for i in range(oof_stack.shape[1])])
oof_proba["ensemble_rank"] = oof_ranks.mean(axis=1) / len(y_tv)


# --- Threshold search ---
def find_best_threshold(y_true, proba, recall_floor=RECALL_FLOOR):
    thresholds = np.linspace(0.01, 0.99, 99)
    best = None
    for t in thresholds:
        pred = (proba >= t).astype(int)
        recall_subs = recall_score(y_true, pred, pos_label=1, zero_division=0)
        if recall_subs < recall_floor:
            continue
        macro_f1 = f1_score(y_true, pred, average="macro", zero_division=0)
        if best is None or macro_f1 > best["macro_f1"]:
            best = {"threshold": t, "macro_f1": macro_f1, "recall_subs": recall_subs}
    return best


print("\nFinding optimal threshold per classifier...\n")
print(f"{'Classifier':<22} {'Threshold':>10} {'Fold-σ':>8} {'OOF macro-F1':>13} {'OOF subs-recall':>16}")
print("-" * 75)

threshold_results = {}
for name, proba in oof_proba.items():
    best = find_best_threshold(y_tv, proba)
    if best is None:
        threshold_results[name] = {"threshold": None, "macro_f1": None, "recall_subs": None, "fold_std": None}
        print(f"{name:<22} {'FAIL':>10} {'-':>8} {'-':>13} {'-':>16}  (no threshold meets recall ≥ {RECALL_FLOOR})")
        continue

    fold_ts = []
    for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(np.zeros(len(y_tv)), y_tv)):
        fold_proba = proba[te_idx]
        fold_y = y_tv[te_idx]
        fold_best = find_best_threshold(fold_y, fold_proba)
        if fold_best is not None:
            fold_ts.append(fold_best["threshold"])
    fold_std = np.std(fold_ts) if fold_ts else 0

    threshold_results[name] = {
        "threshold": best["threshold"],
        "macro_f1": best["macro_f1"],
        "recall_subs": best["recall_subs"],
        "fold_std": fold_std,
    }
    print(f"{name:<22} {best['threshold']:>10.3f} {fold_std:>8.3f} {best['macro_f1']:>13.4f} {best['recall_subs']:>16.4f}")

for name in threshold_results:
    if name in classifier_results:
        classifier_results[name]["threshold"] = threshold_results[name]["threshold"]
        classifier_results[name]["oof_macro_f1"] = threshold_results[name]["macro_f1"]
        classifier_results[name]["oof_subs_recall"] = threshold_results[name]["recall_subs"]
        classifier_results[name]["threshold_fold_std"] = threshold_results[name]["fold_std"]

with open(CLASSIFIERS_CACHE, "wb") as f:
    pickle.dump(classifier_results, f)

print(f"\n✅ Thresholds saved to {CLASSIFIERS_CACHE}")

Cross-validation pool: 1,878 sentences (train + val)
  SUBS: 1660  BOIL: 218

Loaded 7 cached OOF probabilities: ['logreg_emb', 'svm_rbf_emb', 'histgbm_combined', 'mlp_emb', 'tfidf_ngram', 'floret_real', 'finbert_finetuned']
  CV: logreg_emb (cached, skipping)
  CV: svm_rbf_emb (cached, skipping)
  CV: histgbm_combined (cached, skipping)
  CV: mlp_emb (cached, skipping)
  CV: tfidf_ngram (cached, skipping)
  CV: floret_real (cached, skipping)
  CV: finbert_finetuned (cached, skipping)

✅ All OOF probabilities cached in C:\Users\loren\Desktop\MTH9796\HW2\cache\oof_probas.npz

Finding optimal threshold per classifier...

Classifier              Threshold   Fold-σ  OOF macro-F1  OOF subs-recall
---------------------------------------------------------------------------
logreg_emb                  0.240    0.073        0.8653           0.9789
svm_rbf_emb                 0.340    0.172        0.8741           0.9928
histgbm_combined            0.870    0.220        0.8663           0.9813
m

In [36]:
# Cell 24b: Refit base models on train+val and overwrite test_proba
#
# The threshold tuned in Cell 24 was calibrated against 5-fold OOF probabilities,
# where each fold trains on ~80% of train+val. The deployed bundle (Cell 26b)
# refits on ALL of train+val. To keep the reported leaderboard consistent with
# both the threshold's calibration and the deployed model, we regenerate
# test_proba from train+val-fitted models here.
#
# The test set is still untouched for fitting — we only PREDICT on it.

import numpy as np

print("Refitting base models on train+val and regenerating test_proba...")

trainval_mask = (df_split["split"] == "train") | (df_split["split"] == "val")
test_mask = df_split["split"] == "test"

X_emb_tv = X_emb[trainval_mask.values]
X_combined_tv = X_combined[trainval_mask.values]
texts_tv = df_split.loc[trainval_mask, "text"].values
y_tv = df_split.loc[trainval_mask, "y"].values

X_emb_test = X_emb[test_mask.values]
X_combined_test = X_combined[test_mask.values]
texts_test = df_split.loc[test_mask, "text"].values


# --- Cache helpers (self-contained) ---
REFIT_CACHE = CACHE_DIR / "refit_test_probas.npz"


def load_refit_cache():
    if REFIT_CACHE.exists():
        d = np.load(REFIT_CACHE)
        return {k: d[k] for k in d.files}
    return {}


def save_refit_cache(refit_dict):
    np.savez(REFIT_CACHE, **refit_dict)


refit_cache = load_refit_cache()
print(f"Loaded {len(refit_cache)} cached refit test_probas: {list(refit_cache.keys())}")


def get_or_refit(name, fn):
    """Use cached test_proba if available, else compute and cache."""
    if name in refit_cache:
        print(f"  {name} (cached, skipping refit)")
        classifier_results[name]["test_proba"] = refit_cache[name]
        return
    print(f"  {name} (refitting on train+val...)")
    proba = fn()
    classifier_results[name]["test_proba"] = proba
    refit_cache[name] = proba
    save_refit_cache(refit_cache)


# --- Refit each base model on train+val and predict on test ---

get_or_refit("logreg_emb", lambda: LogisticRegression(
    C=1.0, max_iter=2000, class_weight="balanced", random_state=RANDOM_SEED
).fit(X_emb_tv, y_tv).predict_proba(X_emb_test)[:, 1])

get_or_refit("svm_rbf_emb", lambda: SVC(
    kernel="rbf", C=1.0, gamma="scale", probability=True,
    class_weight="balanced", random_state=RANDOM_SEED
).fit(X_emb_tv, y_tv).predict_proba(X_emb_test)[:, 1])


def _refit_histgbm():
    sw = compute_sample_weight("balanced", y_tv)
    m = HistGradientBoostingClassifier(
        max_iter=300, max_leaf_nodes=31, learning_rate=0.1, random_state=RANDOM_SEED
    )
    m.fit(X_combined_tv, y_tv, sample_weight=sw)
    return m.predict_proba(X_combined_test)[:, 1]
get_or_refit("histgbm_combined", _refit_histgbm)


get_or_refit("mlp_emb", lambda: MLPClassifier(
    hidden_layer_sizes=(128, 32), activation="relu", solver="adam",
    alpha=1e-4, batch_size=64, learning_rate_init=1e-3, max_iter=200,
    early_stopping=True, validation_fraction=0.15, random_state=RANDOM_SEED
).fit(X_emb_tv, y_tv).predict_proba(X_emb_test)[:, 1])


def _refit_tfidf():
    p = Pipeline([
        ("tfidf", TfidfVectorizer(analyzer="word", ngram_range=(1, 2),
                                  min_df=2, max_df=0.95, sublinear_tf=True)),
        ("lr", LogisticRegression(C=1.0, max_iter=2000, class_weight="balanced",
                                  random_state=RANDOM_SEED)),
    ])
    p.fit(texts_tv, y_tv)
    return p.predict_proba(texts_test)[:, 1]
get_or_refit("tfidf_ngram", _refit_tfidf)


get_or_refit("floret_real", lambda: fit_predict_floret(texts_tv, y_tv, texts_test))


get_or_refit("finbert_finetuned", lambda: fit_predict_finbert(texts_tv, y_tv, texts_test))


# --- Rebuild ensemble test_probas from the refitted members ---
ENSEMBLE_MEMBERS = ["svm_rbf_emb", "mlp_emb", "tfidf_ngram", "histgbm_combined", "logreg_emb"]
test_stack = np.column_stack([classifier_results[m]["test_proba"] for m in ENSEMBLE_MEMBERS])

# Mean-prob ensemble
classifier_results["ensemble_mean"]["test_proba"] = test_stack.mean(axis=1)

# Rank-averaged ensemble — calibrated against OOF reference distributions.
# Use the cached OOF probas from Cell 24 — do NOT recompute (would take ~2 hours
# for FinBERT, and we don't need FinBERT here anyway since it's not an ensemble member).
ref_dists = {m: np.sort(oof_proba[m]) for m in ENSEMBLE_MEMBERS}

test_percentiles = np.column_stack([
    np.searchsorted(ref_dists[m], classifier_results[m]["test_proba"], side="right") / len(ref_dists[m])
    for m in ENSEMBLE_MEMBERS
])
classifier_results["ensemble_rank"]["test_proba"] = test_percentiles.mean(axis=1)

# Persist
with open(CLASSIFIERS_CACHE, "wb") as f:
    pickle.dump(classifier_results, f)

print(f"\n✅ test_proba regenerated for all base models and ensembles using train+val fits.")
print(f"   Threshold τ stays as tuned in Cell 24 (recall floor + macro-F1 maximization on OOF).")
print(f"   Re-run Cell 25 to see the corrected leaderboard.")

Refitting base models on train+val and regenerating test_proba...
Loaded 7 cached refit test_probas: ['logreg_emb', 'svm_rbf_emb', 'histgbm_combined', 'mlp_emb', 'tfidf_ngram', 'floret_real', 'finbert_finetuned']
  logreg_emb (cached, skipping refit)
  svm_rbf_emb (cached, skipping refit)
  histgbm_combined (cached, skipping refit)
  mlp_emb (cached, skipping refit)
  tfidf_ngram (cached, skipping refit)
  floret_real (cached, skipping refit)
  finbert_finetuned (cached, skipping refit)

✅ test_proba regenerated for all base models and ensembles using train+val fits.
   Threshold τ stays as tuned in Cell 24 (recall floor + macro-F1 maximization on OOF).
   Re-run Cell 25 to see the corrected leaderboard.


In [37]:
# Cell 25: Final test-set evaluation with tuned thresholds + full leaderboard

print("=" * 90)
print("FINAL TEST EVALUATION (using OOF-tuned thresholds)")
print("=" * 90)

# Build the leaderboard table
leaderboard_rows = []
for name, res in classifier_results.items():
    t = res.get("threshold")
    if t is None:
        # Fallback for rules_only (no threshold tuning needed)
        t = 0.5
    
    val_pred = (res["val_proba"] >= t).astype(int)
    test_pred = (res["test_proba"] >= t).astype(int)
    
    y_val = splits["val"]["y"]
    y_test = splits["test"]["y"]
    
    leaderboard_rows.append({
        "classifier": name,
        "threshold": round(t, 3),
        "fold_std": round(res.get("threshold_fold_std", 0) or 0, 3),
        "test_acc": round(accuracy_score(y_test, test_pred), 3),
        "test_macro_f1": round(f1_score(y_test, test_pred, average="macro"), 3),
        "test_subs_p": round(precision_score(y_test, test_pred, pos_label=1), 3),
        "test_subs_r": round(recall_score(y_test, test_pred, pos_label=1), 3),
        "test_subs_f1": round(f1_score(y_test, test_pred, pos_label=1), 3),
        "test_boil_p": round(precision_score(y_test, test_pred, pos_label=0, zero_division=0), 3),
        "test_boil_r": round(recall_score(y_test, test_pred, pos_label=0, zero_division=0), 3),
        "test_boil_f1": round(f1_score(y_test, test_pred, pos_label=0, zero_division=0), 3),
        "train_time_s": round(res.get("train_time_sec", 0), 2),
        "throughput": int(res.get("throughput_sent_per_sec", 0)),
    })

df_leaderboard = pd.DataFrame(leaderboard_rows).sort_values("test_macro_f1", ascending=False)

print("\n=== LEADERBOARD (sorted by test macro-F1) ===\n")
print(df_leaderboard.to_string(index=False))

# Save the leaderboard
LEADERBOARD_CACHE = MODELS_DIR / "leaderboard.csv"
df_leaderboard.to_csv(LEADERBOARD_CACHE, index=False)
print(f"\n✅ Leaderboard saved → {LEADERBOARD_CACHE}")

# --- Pick the winner that satisfies the recall floor on TEST ---
# Per rubric: among classifiers that pass test subs-recall ≥ 0.96, pick highest macro-F1
RECALL_FLOOR = 0.96
feasible = df_leaderboard[df_leaderboard["test_subs_r"] >= RECALL_FLOOR]

print(f"\n=== Models passing test substantive-recall ≥ {RECALL_FLOOR} ===")
if len(feasible) == 0:
    print("⚠️  NO MODEL meets the recall floor on test. Will use highest-recall model.")
    winner = df_leaderboard.iloc[df_leaderboard["test_subs_r"].argmax()]
else:
    print(feasible[["classifier", "threshold", "test_macro_f1", "test_subs_r", "test_boil_f1"]].to_string(index=False))
    winner = feasible.iloc[0]

print(f"\n🏆 WINNER: {winner['classifier']}")
print(f"   Threshold: {winner['threshold']}")
print(f"   Test macro-F1: {winner['test_macro_f1']}")
print(f"   Test subs-recall: {winner['test_subs_r']}")
print(f"   Test boil-F1: {winner['test_boil_f1']}")

# --- Confusion matrix for the winner ---
winner_name = winner["classifier"]
winner_threshold = winner["threshold"]
y_test = splits["test"]["y"]
test_pred = (classifier_results[winner_name]["test_proba"] >= winner_threshold).astype(int)

print(f"\n=== Confusion matrix for {winner_name} on test ===")
cm = confusion_matrix(y_test, test_pred, labels=[1, 0])  # SUBS=1, BOIL=0
print(f"           pred=SUBS  pred=BOIL")
print(f"true=SUBS    {cm[0,0]:5d}      {cm[0,1]:5d}")
print(f"true=BOIL    {cm[1,0]:5d}      {cm[1,1]:5d}")

print(f"\n=== Classification report for {winner_name} on test ===")
print(classification_report(y_test, test_pred, target_names=["BOILERPLATE", "SUBSTANTIVE"], digits=3))

FINAL TEST EVALUATION (using OOF-tuned thresholds)

=== LEADERBOARD (sorted by test macro-F1) ===

       classifier  threshold  fold_std  test_acc  test_macro_f1  test_subs_p  test_subs_r  test_subs_f1  test_boil_p  test_boil_r  test_boil_f1  train_time_s  throughput
finbert_finetuned       0.28     0.070     0.964          0.909        0.976        0.983         0.980        0.863        0.815         0.838       1308.21           6
       logreg_emb       0.24     0.073     0.962          0.894        0.963        0.995         0.979        0.950        0.704         0.809          0.01      548275
    ensemble_rank       0.12     0.015     0.957          0.884        0.963        0.990         0.976        0.905        0.704         0.792          0.00      194056
    ensemble_mean       0.51     0.087     0.957          0.882        0.960        0.993         0.976        0.925        0.685         0.787          0.00      798268
      svm_rbf_emb       0.34     0.172     0.951   

In [38]:
'''
# Cell 26: Save everything the GUI needs to load
#
# We save a "model bundle" — a single pickle containing all the pieces needed to
# classify a new sentence end-to-end:
#   - the embedding model (sentence-transformers)
#   - the regex feature extractor (functions + patterns)
#   - the trained ensemble member models
#   - the rank-averaged ensemble logic
#   - the tuned threshold (0.12)
#
# The GUI loads this bundle and calls a single classify() function.

import pickle
from pathlib import Path

WINNER_BUNDLE_PATH = MODELS_DIR / "winner_bundle.pkl"

# We need to retrain the ensemble members on the FULL train+val data
# (so we use all available labeled data, not just the 60% train slice).
# Test set stays frozen and is not used for fitting anything.

print("Refitting ensemble members on full train+val pool...")

trainval_mask = (df_split["split"] == "train") | (df_split["split"] == "val")
X_emb_full = X_emb[trainval_mask.values]
X_combined_full = X_combined[trainval_mask.values]
texts_full = df_split.loc[trainval_mask, "text"].values
y_full = df_split.loc[trainval_mask, "y"].values

print(f"  Training pool: {len(y_full):,} sentences")

# Refit each ensemble member
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_sample_weight

print("  Fitting LogReg...")
m_logreg = LogisticRegression(C=1.0, max_iter=2000, class_weight="balanced", random_state=RANDOM_SEED)
m_logreg.fit(X_emb_full, y_full)

print("  Fitting SVM-RBF...")
m_svm = SVC(kernel="rbf", C=1.0, gamma="scale", probability=True, class_weight="balanced", random_state=RANDOM_SEED)
m_svm.fit(X_emb_full, y_full)

print("  Fitting HistGBM...")
sw = compute_sample_weight("balanced", y_full)
m_histgbm = HistGradientBoostingClassifier(max_iter=300, max_leaf_nodes=31, learning_rate=0.1, random_state=RANDOM_SEED)
m_histgbm.fit(X_combined_full, y_full, sample_weight=sw)

print("  Fitting MLP...")
m_mlp = MLPClassifier(hidden_layer_sizes=(128, 32), activation="relu", solver="adam",
                      alpha=1e-4, batch_size=64, learning_rate_init=1e-3, max_iter=200,
                      early_stopping=True, validation_fraction=0.15, random_state=RANDOM_SEED)
m_mlp.fit(X_emb_full, y_full)

print("  Fitting TF-IDF+LogReg...")
m_tfidf = Pipeline([
    ("tfidf", TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True)),
    ("lr", LogisticRegression(C=1.0, max_iter=2000, class_weight="balanced", random_state=RANDOM_SEED)),
])
m_tfidf.fit(texts_full, y_full)

# --- Build the bundle ---
bundle = {
    # Models
    "models": {
        "logreg_emb": m_logreg,
        "svm_rbf_emb": m_svm,
        "histgbm_combined": m_histgbm,
        "mlp_emb": m_mlp,
        "tfidf_ngram": m_tfidf,
    },
    # Inference config
    "ensemble_members": ["svm_rbf_emb", "mlp_emb", "tfidf_ngram", "histgbm_combined", "logreg_emb"],
    "threshold": 0.12,  # tuned via OOF, recall floor 0.96
    "ensemble_type": "rank_average",
    # Feature pipeline
    "regex_patterns_boil": {k: v.pattern for k, v in PATTERNS_BOIL.items()},
    "regex_patterns_subs": {k: v.pattern for k, v in PATTERNS_SUBS.items()},
    "regex_feature_order": list(df_features_regex.drop(columns=["sentence_id"]).columns),
    # Embedding model name (we'll re-load it in the GUI to keep the bundle small)
    "embedding_model_name": "sentence-transformers/all-MiniLM-L6-v2",
    # Metadata
    "training_size": int(len(y_full)),
    "training_class_balance": {
        "SUBSTANTIVE": int((y_full == 1).sum()),
        "BOILERPLATE": int((y_full == 0).sum()),
    },
    "test_metrics": {
        "macro_f1": 0.872,
        "subs_recall": 0.988,
        "boil_f1": 0.771,
        "accuracy": 0.953,
    },
}

with open(WINNER_BUNDLE_PATH, "wb") as f:
    pickle.dump(bundle, f)

import os
size_mb = os.path.getsize(WINNER_BUNDLE_PATH) / 1e6
print(f"\n✅ Saved winner bundle → {WINNER_BUNDLE_PATH}")
print(f"   Size: {size_mb:.1f} MB")
print(f"   Threshold: {bundle['threshold']}")
print(f"   Ensemble type: {bundle['ensemble_type']}")
print(f"   Members: {bundle['ensemble_members']}")
'''

'\n# Cell 26: Save everything the GUI needs to load\n#\n# We save a "model bundle" — a single pickle containing all the pieces needed to\n# classify a new sentence end-to-end:\n#   - the embedding model (sentence-transformers)\n#   - the regex feature extractor (functions + patterns)\n#   - the trained ensemble member models\n#   - the rank-averaged ensemble logic\n#   - the tuned threshold (0.12)\n#\n# The GUI loads this bundle and calls a single classify() function.\n\nimport pickle\nfrom pathlib import Path\n\nWINNER_BUNDLE_PATH = MODELS_DIR / "winner_bundle.pkl"\n\n# We need to retrain the ensemble members on the FULL train+val data\n# (so we use all available labeled data, not just the 60% train slice).\n# Test set stays frozen and is not used for fitting anything.\n\nprint("Refitting ensemble members on full train+val pool...")\n\ntrainval_mask = (df_split["split"] == "train") | (df_split["split"] == "val")\nX_emb_full = X_emb[trainval_mask.values]\nX_combined_full = X_combined[t

In [39]:
'''
# Cell 27: Define and test the inference function

import re
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import rankdata


def load_bundle(bundle_path):
    """Load the winner bundle from disk."""
    with open(bundle_path, "rb") as f:
        return pickle.load(f)


def make_regex_extractor(bundle):
    """Build the same regex feature extractor used at training time, from the bundle's patterns."""
    pat_boil = {k: re.compile(v) for k, v in bundle["regex_patterns_boil"].items()}
    pat_subs = {k: re.compile(v) for k, v in bundle["regex_patterns_subs"].items()}
    feature_order = bundle["regex_feature_order"]
    
    def extract(text):
        feats = {}
        for name, pat in pat_boil.items():
            feats[name] = int(bool(pat.search(text)))
        for name, pat in pat_subs.items():
            feats[name] = int(bool(pat.search(text)))
        feats["f_char_len"] = len(text)
        feats["f_word_count"] = len(text.split())
        feats["f_digit_count"] = sum(c.isdigit() for c in text)
        feats["f_dollar_count"] = text.count("$")
        feats["f_question_mark"] = int("?" in text)
        feats["f_starts_capital"] = int(text[0].isupper() if text else 0)
        feats["f_uppercase_ratio"] = sum(c.isupper() for c in text) / max(1, len(text))
        feats["f_punct_ratio"] = sum(1 for c in text if c in ".,;:!?") / max(1, len(text))
        # Return in the order the model was trained on
        return np.array([feats[k] for k in feature_order], dtype=np.float32)
    
    return extract


def classify_sentences(texts, bundle, embedding_model):
    """Classify a list of sentences. Returns list of dicts with text, label, prob, threshold."""
    if not texts:
        return []
    
    # Embeddings
    X_emb = embedding_model.encode(
        texts,
        batch_size=32,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    
    # Regex features
    extractor = make_regex_extractor(bundle)
    X_regex = np.array([extractor(t) for t in texts], dtype=np.float32)
    X_combined = np.hstack([X_emb, X_regex])
    
    # Run each ensemble member
    member_probas = {}
    member_probas["logreg_emb"] = bundle["models"]["logreg_emb"].predict_proba(X_emb)[:, 1]
    member_probas["svm_rbf_emb"] = bundle["models"]["svm_rbf_emb"].predict_proba(X_emb)[:, 1]
    member_probas["histgbm_combined"] = bundle["models"]["histgbm_combined"].predict_proba(X_combined)[:, 1]
    member_probas["mlp_emb"] = bundle["models"]["mlp_emb"].predict_proba(X_emb)[:, 1]
    member_probas["tfidf_ngram"] = bundle["models"]["tfidf_ngram"].predict_proba(texts)[:, 1]
    
    # Rank-average ensemble
    members = bundle["ensemble_members"]
    member_stack = np.column_stack([member_probas[m] for m in members])
    ranks = np.column_stack([rankdata(member_stack[:, i]) for i in range(member_stack.shape[1])])
    proba = ranks.mean(axis=1) / len(texts) if len(texts) > 1 else np.array([0.5])
    
    # Apply threshold
    threshold = bundle["threshold"]
    labels = ["SUBSTANTIVE" if p >= threshold else "BOILERPLATE" for p in proba]
    
    return [
        {"text": t, "label": l, "probability": float(p), "threshold": threshold}
        for t, l, p in zip(texts, labels, proba)
    ]


# --- Test it on 8 sample sentences (4 known boilerplate, 4 known substantive) ---
print("Loading bundle and embedding model...")
bundle = load_bundle(WINNER_BUNDLE_PATH)
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer(bundle["embedding_model_name"])

test_sentences = [
    # Expected boilerplate
    "The next question comes from CJ Muse with Cantor Fitzgerald.",
    "Thank you for joining today's call.",
    "Please refer to our 10-Q for forward-looking statements.",
    "I'll now turn the call over to our CFO.",
    # Expected substantive
    "Revenue grew 38% year-over-year to $2.1 billion in the quarter.",
    "We're guiding gross margin in the range of 75 to 76 percent for Q3.",
    "Our Data Center segment contributed $1.4 billion this quarter.",
    "Can you walk us through the puts and takes on operating margin for next year?",
]

print(f"\nClassifying {len(test_sentences)} test sentences...\n")
results = classify_sentences(test_sentences, bundle, embedding_model)

for r in results:
    icon = "🟢" if r["label"] == "SUBSTANTIVE" else "🔴"
    print(f"{icon}  [{r['label']:11s}  p={r['probability']:.3f}]  {r['text']}")

print(f"\nThreshold: {bundle['threshold']}")

'''

'\n# Cell 27: Define and test the inference function\n\nimport re\nimport pickle\nimport numpy as np\nimport pandas as pd\nfrom pathlib import Path\nfrom scipy.stats import rankdata\n\n\ndef load_bundle(bundle_path):\n    """Load the winner bundle from disk."""\n    with open(bundle_path, "rb") as f:\n        return pickle.load(f)\n\n\ndef make_regex_extractor(bundle):\n    """Build the same regex feature extractor used at training time, from the bundle\'s patterns."""\n    pat_boil = {k: re.compile(v) for k, v in bundle["regex_patterns_boil"].items()}\n    pat_subs = {k: re.compile(v) for k, v in bundle["regex_patterns_subs"].items()}\n    feature_order = bundle["regex_feature_order"]\n\n    def extract(text):\n        feats = {}\n        for name, pat in pat_boil.items():\n            feats[name] = int(bool(pat.search(text)))\n        for name, pat in pat_subs.items():\n            feats[name] = int(bool(pat.search(text)))\n        feats["f_char_len"] = len(text)\n        feats["f_wo

In [40]:
# Cell 26b: Build the deployment bundle
#
# Two-track strategy:
#   - Leaderboard winner: finbert_finetuned (macro-F1 0.909, recall 0.983)
#   - Deployed for GUI:   logreg_emb        (macro-F1 0.894, recall 0.995)
#
# Why deploy logreg instead of the leaderboard winner:
# FinBERT's CPU inference throughput is ~2-7 sentences/second (depending on
# hardware), which means classifying a typical 350-sentence earnings call
# takes 1-3 minutes. The handout requires the GUI to "run in under a minute".
# Logreg classifies the same transcript in milliseconds.
#
# We deploy logreg because:
#   1. It satisfies the recall floor (0.995, well above 0.96).
#   2. Its macro-F1 (0.894) is within 0.015 of FinBERT's (0.909).
#   3. Its inference is ~50,000x faster, making the GUI usable.
#
# The fine-tuned FinBERT remains on disk at models/finbert_deployed/ for
# reproducibility and for any user who wants to swap it in for higher
# accuracy at the cost of latency.

import pickle
from pathlib import Path
from sklearn.linear_model import LogisticRegression

WINNER_BUNDLE_PATH = MODELS_DIR / "winner_bundle.pkl"

trainval_mask = (df_split["split"] == "train") | (df_split["split"] == "val")
X_emb_tv = X_emb[trainval_mask.values]
y_tv = df_split.loc[trainval_mask, "y"].values

print(f"Refitting logreg_emb on full train+val pool ({len(y_tv)} sentences)...")

m_logreg = LogisticRegression(
    C=1.0, max_iter=2000, class_weight="balanced", random_state=RANDOM_SEED
)
m_logreg.fit(X_emb_tv, y_tv)

bundle = {
    "model": m_logreg,
    "threshold": 0.24,  # tuned via 5-fold OOF, recall floor 0.96
    "model_type": "logreg_emb",
    "embedding_model_name": "sentence-transformers/all-MiniLM-L6-v2",
    "training_size": int(len(y_tv)),
    "training_class_balance": {
        "SUBSTANTIVE": int((y_tv == 1).sum()),
        "BOILERPLATE": int((y_tv == 0).sum()),
    },
    "test_metrics": {
        "macro_f1": 0.894,
        "subs_recall": 0.995,
        "boil_f1": 0.809,
        "accuracy": 0.962,
    },
    "leaderboard_winner_note": (
        "Leaderboard winner is finbert_finetuned (macro-F1 0.909). "
        "Deployed logreg here for GUI latency reasons; see write-up §7. "
        "Fine-tuned FinBERT is available at models/finbert_deployed/ for "
        "users who prefer accuracy over speed."
    ),
}

with open(WINNER_BUNDLE_PATH, "wb") as f:
    pickle.dump(bundle, f)

print(f"\n✅ Deployment bundle saved → {WINNER_BUNDLE_PATH}")
print(f"   Deployed model: logistic regression on 384-dim sentence embeddings")
print(f"   Threshold: 0.24")
print(f"   Bundle size: {WINNER_BUNDLE_PATH.stat().st_size / 1024:.1f} KB")
print(f"   FinBERT (leaderboard winner) preserved at: {MODELS_DIR / 'finbert_deployed'}")

Refitting logreg_emb on full train+val pool (1878 sentences)...

✅ Deployment bundle saved → C:\Users\loren\Desktop\MTH9796\HW2\models\winner_bundle.pkl
   Deployed model: logistic regression on 384-dim sentence embeddings
   Threshold: 0.24
   Bundle size: 4.2 KB
   FinBERT (leaderboard winner) preserved at: C:\Users\loren\Desktop\MTH9796\HW2\models\finbert_deployed


In [41]:
# Cell 27: Inference function for the deployed logreg model

import pickle
import numpy as np
from pathlib import Path


def load_bundle(bundle_path):
    with open(bundle_path, "rb") as f:
        return pickle.load(f)


def classify_sentences(texts, bundle, embedding_model):
    """Classify a list of sentences using the deployed logreg model."""
    if not texts:
        return []

    X_emb = embedding_model.encode(
        texts,
        batch_size=32,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    proba = bundle["model"].predict_proba(X_emb)[:, 1]  # P(SUBSTANTIVE)
    threshold = bundle["threshold"]
    labels = ["SUBSTANTIVE" if p >= threshold else "BOILERPLATE" for p in proba]

    return [
        {"text": t, "label": l, "probability": float(p), "threshold": threshold}
        for t, l, p in zip(texts, labels, proba)
    ]


# Sanity test
print("Loading bundle and embedding model...")
bundle = load_bundle(WINNER_BUNDLE_PATH)
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer(bundle["embedding_model_name"])

test_sentences = [
    "The next question comes from CJ Muse with Cantor Fitzgerald.",
    "Thank you for joining today's call.",
    "Please refer to our 10-Q for forward-looking statements.",
    "I'll now turn the call over to our CFO.",
    "Revenue grew 38% year-over-year to $2.1 billion in the quarter.",
    "We're guiding gross margin in the range of 75 to 76 percent for Q3.",
    "Our Data Center segment contributed $1.4 billion this quarter.",
    "Can you walk us through the puts and takes on operating margin for next year?",
]

print("\nBatch test:\n")
for r in classify_sentences(test_sentences, bundle, embedding_model):
    icon = "🟢" if r["label"] == "SUBSTANTIVE" else "🔴"
    print(f"{icon}  [{r['label']:11s} p={r['probability']:.3f}]  {r['text']}")

# One-at-a-time test (should give identical results — logreg is stateless per-sentence)
print("\nOne-at-a-time test (should match batch results):\n")
for s in test_sentences:
    r = classify_sentences([s], bundle, embedding_model)[0]
    icon = "🟢" if r["label"] == "SUBSTANTIVE" else "🔴"
    print(f"{icon}  [{r['label']:11s} p={r['probability']:.3f}]  {r['text']}")

Loading bundle and embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6525.39it/s]



Batch test:

🔴  [BOILERPLATE p=0.010]  The next question comes from CJ Muse with Cantor Fitzgerald.
🔴  [BOILERPLATE p=0.017]  Thank you for joining today's call.
🔴  [BOILERPLATE p=0.059]  Please refer to our 10-Q for forward-looking statements.
🔴  [BOILERPLATE p=0.064]  I'll now turn the call over to our CFO.
🟢  [SUBSTANTIVE p=0.980]  Revenue grew 38% year-over-year to $2.1 billion in the quarter.
🟢  [SUBSTANTIVE p=0.850]  We're guiding gross margin in the range of 75 to 76 percent for Q3.
🟢  [SUBSTANTIVE p=0.989]  Our Data Center segment contributed $1.4 billion this quarter.
🟢  [SUBSTANTIVE p=0.821]  Can you walk us through the puts and takes on operating margin for next year?

One-at-a-time test (should match batch results):

🔴  [BOILERPLATE p=0.010]  The next question comes from CJ Muse with Cantor Fitzgerald.
🔴  [BOILERPLATE p=0.017]  Thank you for joining today's call.
🔴  [BOILERPLATE p=0.059]  Please refer to our 10-Q for forward-looking statements.
🔴  [BOILERPLATE p=0.064]  I'

In [42]:
# Quick sanity check that inference.py works as a module
import sys
from importlib import reload

# Add current dir to path so we can import the module
if "" not in sys.path:
    sys.path.insert(0, "")

import inference
reload(inference)

bundle, model = inference.load_pipeline("models/winner_bundle.pkl")
results = inference.classify_sentences([
    "The next question comes from CJ Muse with Cantor Fitzgerald.",
    "Revenue grew 38% year-over-year to $2.1 billion in the quarter.",
], bundle, model)
for r in results:
    print(f"  [{r['label']}  p={r['probability']:.3f}]  {r['text']}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5714.54it/s]


  [BOILERPLATE  p=0.010]  The next question comes from CJ Muse with Cantor Fitzgerald.
  [SUBSTANTIVE  p=0.980]  Revenue grew 38% year-over-year to $2.1 billion in the quarter.


In [43]:
# Cell 28: Error analysis on the held-out test set
#
# We collect all misclassifications (with the WINNING model and tuned threshold)
# from the frozen test set, separate them by direction (false positives vs false negatives),
# and print examples grouped by what we think is causing each failure mode.
#
# This is the source material for the "Error analysis" section of the write-up.

import pickle
import numpy as np
import pandas as pd

# Load winner predictions (rank-averaged ensemble at threshold 0.12)
WINNER_NAME = "logreg_emb"
WINNER_THRESHOLD = 0.24

test_proba = classifier_results[WINNER_NAME]["test_proba"]
test_pred = (test_proba >= WINNER_THRESHOLD).astype(int)
y_test = splits["test"]["y"]
test_texts = splits["test"]["text"]
test_ids = splits["test"]["sentence_id"]

# Build error dataframe
errors_df = pd.DataFrame({
    "sentence_id": test_ids,
    "text": test_texts,
    "true_label": ["SUBSTANTIVE" if y == 1 else "BOILERPLATE" for y in y_test],
    "pred_label": ["SUBSTANTIVE" if p == 1 else "BOILERPLATE" for p in test_pred],
    "probability": test_proba,
})

# Add metadata
errors_df = errors_df.merge(
    df_split[["sentence_id", "section", "speaker_role", "claude_haiku", "gpt4o_mini", "mistral_small"]],
    on="sentence_id",
    how="left",
)

# Filter to misclassifications only
mistakes = errors_df[errors_df["true_label"] != errors_df["pred_label"]].copy()
mistakes["error_type"] = mistakes.apply(
    lambda r: f"{r.true_label[:4]} → predicted {r.pred_label[:4]}", axis=1
)

print(f"=== Error summary on test set ===")
print(f"Total test sentences: {len(y_test)}")
print(f"Total misclassifications: {len(mistakes)}")
print(f"  False BOIL (gold=SUBS, pred=BOIL): {((errors_df['true_label']=='SUBSTANTIVE') & (errors_df['pred_label']=='BOILERPLATE')).sum()}")
print(f"  False SUBS (gold=BOIL, pred=SUBS): {((errors_df['true_label']=='BOILERPLATE') & (errors_df['pred_label']=='SUBSTANTIVE')).sum()}")
print()

# Helper: did all 3 LLM judges agree with the gold label?
def judge_unanimity(row):
    judges = [row["claude_haiku"], row["gpt4o_mini"], row["mistral_small"]]
    if all(j == row["true_label"] for j in judges):
        return "unanimous_judges_agree"
    elif all(j == row["pred_label"] for j in judges):
        return "judges_disagree_with_gold"  # likely label noise
    else:
        return "judges_were_split"


mistakes["judge_status"] = mistakes.apply(judge_unanimity, axis=1)

print(f"=== Judge unanimity on errors ===")
print("(Tells us whether the 'mistake' is a real model error or possible label noise)")
print(mistakes["judge_status"].value_counts().to_string())
print()

# Sort by probability for readability
mistakes_sorted = mistakes.sort_values(["error_type", "probability"])

# --- False SUBS predictions (gold=BOIL, pred=SUBS) — HIGH-COST: pollutes substantive content ---
print("=" * 100)
print("FALSE SUBSTANTIVE PREDICTIONS (gold=BOIL, model said SUBS)")
print("These hurt downstream: a boilerplate sentence got forwarded as if it were substantive.")
print("=" * 100)
fs = mistakes_sorted[mistakes_sorted["error_type"] == "BOIL → predicted SUBS"]
for _, r in fs.iterrows():
    judges = f"C={r.claude_haiku[:4]}/G={r.gpt4o_mini[:4]}/M={r.mistral_small[:4]}"
    print(f"\n  [{r.section}/{r.speaker_role}]  prob={r.probability:.3f}  judges: {judges}  status: {r.judge_status}")
    print(f"  → {r.text[:220]}")

# --- False BOIL predictions (gold=SUBS, pred=BOIL) — LOW-COST but reduces recall ---
print("\n" + "=" * 100)
print("FALSE BOILERPLATE PREDICTIONS (gold=SUBS, model said BOIL)")
print("These hurt the substantive-recall floor. The threshold was tuned to keep these rare.")
print("=" * 100)
fb = mistakes_sorted[mistakes_sorted["error_type"] == "SUBS → predicted BOIL"]
for _, r in fb.iterrows():
    judges = f"C={r.claude_haiku[:4]}/G={r.gpt4o_mini[:4]}/M={r.mistral_small[:4]}"
    print(f"\n  [{r.section}/{r.speaker_role}]  prob={r.probability:.3f}  judges: {judges}  status: {r.judge_status}")
    print(f"  → {r.text[:220]}")

# --- Save for the write-up ---
ERRORS_CACHE = MODELS_DIR / "test_errors.parquet"
mistakes.to_parquet(ERRORS_CACHE, index=False)
print(f"\n\n✅ Saved → {ERRORS_CACHE}")

=== Error summary on test set ===
Total test sentences: 470
Total misclassifications: 18
  False BOIL (gold=SUBS, pred=BOIL): 2
  False SUBS (gold=BOIL, pred=SUBS): 16

=== Judge unanimity on errors ===
(Tells us whether the 'mistake' is a real model error or possible label noise)
judge_status
judges_were_split         14
unanimous_judges_agree     4

FALSE SUBSTANTIVE PREDICTIONS (gold=BOIL, model said SUBS)
These hurt downstream: a boilerplate sentence got forwarded as if it were substantive.

  [presentation/executive]  prob=0.251  judges: C=BOIL/G=SUBS/M=SUBS  status: judges_were_split
  → Above all, they remind us of the hard work and the hustle that is required to win.

  [qa/executive]  prob=0.273  judges: C=BOIL/G=BOIL/M=BOIL  status: unanimous_judges_agree
  → Let me just add one more point on this thing, Brian.

  [qa/executive]  prob=0.373  judges: C=BOIL/G=BOIL/M=BOIL  status: unanimous_judges_agree
  → And we intend to talk a lot more about this at our upcoming Investor Da

In [44]:
# Diagnostic: which features actually fire, and how often, in the train+val labeled pool?
import pandas as pd

df_check = df_split.merge(df_features_regex, on="sentence_id", how="inner")
trainval = df_check[df_check["split"].isin(["train", "val"])]

feature_cols = [c for c in df_features_regex.columns if c.startswith("f_")]
firing = (trainval[feature_cols] > 0).mean().sort_values()

print("Feature firing rates (fraction of train+val sentences where feature > 0):")
print(firing.round(4).to_string())

print(f"\nFeatures that fire on < 0.5% of sentences (effectively dead):")
dead = firing[firing < 0.005]
print(dead.round(4).to_string() if len(dead) else "  None")

# Compare per-class firing for the two suspect features
for f in ["f_appreciate_question", "f_thanks_only"]:
    if f in trainval.columns:
        boil_rate = (trainval[trainval["gold_label"] == "BOILERPLATE"][f] > 0).mean()
        subs_rate = (trainval[trainval["gold_label"] == "SUBSTANTIVE"][f] > 0).mean()
        print(f"\n{f}:  BOIL fires={boil_rate:.3%}  SUBS fires={subs_rate:.3%}")

Feature firing rates (fraction of train+val sentences where feature > 0):
f_thanks_only                0.0000
f_replay_recording           0.0000
f_appreciate_question        0.0000
f_congrats                   0.0005
f_closing_remarks            0.0005
f_safe_harbor                0.0027
f_section_transition         0.0059
f_operator_instructions      0.0059
f_call_handoff               0.0064
f_segment_term               0.0069
f_basis_points               0.0075
f_greeting                   0.0112
f_operator_intro             0.0149
f_speaker_intro_with_firm    0.0213
f_margin_term                0.0234
f_year_over_year             0.0293
f_question_mark              0.0548
f_quarter_ref                0.0852
f_guidance_term              0.0873
f_dollar_amount              0.0884
f_dollar_count               0.0884
f_percent                    0.0974
f_growth_decline             0.1230
f_digit_count                0.3126
f_starts_capital             0.9867
f_uppercase_ratio         

In [45]:
import subprocess, sys

# Try installing transformers + torch (CPU is fine — FinBERT inference and fine-tuning are small)
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "transformers>=4.40", "torch", "datasets", "accelerate"],
    capture_output=True, text=True
)
print("STDOUT:", result.stdout[-2000:] if result.stdout else "(empty)")
print("STDERR:", result.stderr[-2000:] if result.stderr else "(empty)")
print("Return code:", result.returncode)

# Verify imports
try:
    import torch
    print(f"\ntorch {torch.__version__}, CUDA available: {torch.cuda.is_available()}")
    import transformers
    print(f"transformers {transformers.__version__}")
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    print("AutoTokenizer + AutoModelForSequenceClassification imported")
    
    # Try loading FinBERT specifically (small download, ~440MB)
    print("\nLoading FinBERT tokenizer (will download on first run)...")
    tok = AutoTokenizer.from_pretrained("ProsusAI/finbert")
    print(f"FinBERT tokenizer loaded. Vocab size: {tok.vocab_size}")
except Exception as e:
    print(f"\n❌ Import or load failed: {type(e).__name__}: {e}")

STDOUT: (empty)
STDERR: (empty)
Return code: 0

torch 2.11.0+cpu, CUDA available: False
transformers 5.7.0
AutoTokenizer + AutoModelForSequenceClassification imported

Loading FinBERT tokenizer (will download on first run)...
FinBERT tokenizer loaded. Vocab size: 30522


In [46]:
from pathlib import Path

caches_to_check = [
    CACHE_DIR / "sentences.parquet",
    CACHE_DIR / "gold_pool.parquet",
    CACHE_DIR / "judge_labels.parquet",
    CACHE_DIR / "splits.parquet",
    CACHE_DIR / "features_regex.parquet",
    CACHE_DIR / "embeddings.npy",
    CACHE_DIR / "finbert_probas.npz",
    CACHE_DIR / "oof_probas.npz",
    CACHE_DIR / "refit_test_probas.npz",
    MODELS_DIR / "classifier_results.pkl",
    MODELS_DIR / "winner_bundle.pkl",
    MODELS_DIR / "finbert_finetuned",
    MODELS_DIR / "finbert_deployed",
]

print(f"{'File':<45} {'Exists':>8} {'Size':>12}")
print("-" * 70)
for p in caches_to_check:
    exists = p.exists()
    if exists:
        if p.is_dir():
            size = sum(f.stat().st_size for f in p.rglob("*") if f.is_file())
        else:
            size = p.stat().st_size
        size_str = f"{size / 1024 / 1024:.1f} MB" if size > 1024 * 1024 else f"{size / 1024:.1f} KB"
    else:
        size_str = "—"
    print(f"{str(p.name):<45} {'✅' if exists else '❌':>8} {size_str:>12}")

File                                            Exists         Size
----------------------------------------------------------------------
sentences.parquet                                    ✅       3.8 MB
gold_pool.parquet                                    ✅     211.9 KB
judge_labels.parquet                                 ✅     191.1 KB
splits.parquet                                       ✅     216.5 KB
features_regex.parquet                               ✅      56.4 KB
embeddings.npy                                       ✅       3.4 MB
finbert_probas.npz                                   ✅       8.4 KB
oof_probas.npz                                       ✅     104.5 KB
refit_test_probas.npz                                ✅      25.6 KB
classifier_results.pkl                               ✅      72.6 KB
winner_bundle.pkl                                    ✅       4.2 KB
finbert_finetuned                                    ✅     418.3 MB
finbert_deployed                             